# LangChain 04 · 中间件、钩子与人工审核

本节把「智能体的横切能力」一次讲透：**人工审核（Human-in-the-loop）**、
**自定义钩子（Hooks）**、**内置中间件全家桶**，再补上官方文档的
**上下文工程总纲**与**自己组装 harness** 两篇延伸。

中间件（Middleware）= 插在智能体循环各环节的可复用逻辑，全部长在钩子上：

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 人工审核 HITL | 危险工具调用前挂起，等人批 approve/reject/edit | `HumanInTheLoopMiddleware` |
| 钩子 Hook | 在 before/after_model 等时机插入自定义逻辑 | `@before_model` / `@wrap_model_call` |
| 内置中间件 | 官方把常用钩子封装成开箱即用的类 | `SummarizationMiddleware` 等 7+5 个 |
| 上下文工程 | 三类控制 × 三种数据源的「总纲」 | `ToolStrategy` / `ToolRuntime` |

> **本 notebook 由 `Agent/02_langchain/` 下 9 个脚本合并而成**：
> `09_人工审核.py` + `09_人工审核_jxsd.py`（人工审核）、
> `10_中间件_钩子.py` + `10_中间件_钩子_jxsd.py`（钩子）、
> `11_内置中间件.py` + `11_内置中间件_jxsd.py` + `11_内置中间件_官方补充.py`（内置中间件）、
> `20_上下文工程_官方补充.py`（上下文工程总纲）、`22_自己组装harness_官方补充.py`（组装 harness）。

**官方文档**
- Middleware：<https://docs.langchain.com/oss/python/langchain/middleware>
- 上下文工程：<https://docs.langchain.com/oss/python/langchain/context-engineering>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用 `.env` 里配置的大模型（当前 `deepseek-flash`） |
| 依赖 | `langchain` / `langgraph` / `deepagents`（venv 已装） |
| 密钥 | `settings.api_key` / `settings.base_url` / `settings.model_name`（已配置） |
| 前置服务 | 无（不依赖数据库 / 外部服务 / Docker） |
| 预计耗时 | 约 2~4 分钟（几十次真实模型调用；第 3.3 节是离线的，见下） |

> **分段说明**：第 1、2、3.1、3.2、4、5 节都要真实模型；
> 第 **3.3 节（`11_内置中间件_官方补充.py`）是全离线**的 —— 它用「脚本模型」顶替真模型，
> **0 次真实模型调用**，断网也能跑，输出逐字节稳定。
> 第 4.2 节（Demo 2 动态切换输出格式）需要端点支持结构化输出：当前 `deepseek-flash`
> 是思考模型、不支持，会打印 `[跳过]` 走降级路径（**这是设计好的，不是失败**，详见 4.2）。

## 本节地图

中间件在智能体执行循环里的位置（`before_agent → [before_model → model → after_model → (tool)]×N → after_agent`）：

```mermaid
graph LR
    A["before_agent<br/>Agent 启动"] --> B["before_model<br/>模型调用前"]
    B --> C["model<br/>真实模型调用"]
    C --> D{"after_model<br/>要调工具吗?"}
    D -->|"是"| E["tool<br/>工具调用"]
    E --> B
    D -->|"否"| F["after_agent<br/>Agent 结束"]
    F --> G["输出"]
```

上面这张图等价于这张表（裸 JupyterLab 不渲染 mermaid，看表即可）：

| 钩子 | 类型 | 触发时机 |
|---|---|---|
| `@before_agent` | 节点式 | Agent 启动前（一次） |
| `@before_model` | 节点式 | 每次模型调用前 |
| `@after_model` | 节点式 | 每次模型响应后 |
| `@after_agent` | 节点式 | Agent 完成后（一次） |
| `@wrap_model_call` | 包裹式 | 包裹模型调用，可重试/短路 |
| `@wrap_tool_call` | 包裹式 | 包裹工具调用，可监控/缓存 |

> 上一节 `03_记忆与流式` 讲「Agent 本身怎么跑」；这一节讲「怎么在跑的各个环节插逻辑」。
> 下一节 `05_多Agent` 会看到多个 Agent 之间怎么交接。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。所以每个 notebook 的第一格
统一向上找仓库根、切过去、塞进 `sys.path`，同时给出 `NB_DIR` / `WORKDIR` 两个变量。
少了这一格，后面每个 `from config import settings` 都会 `ModuleNotFoundError`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

In [ ]:
# ===== 前置条件自检（缺了会打印中文提示）=====
from config import settings

missing = []
if not settings.api_key:
    missing.append("API_KEY")
if not settings.model_name:
    missing.append("MODEL_NAME")
if not settings.base_url:
    missing.append("BASE_URL")

if missing:
    print(f"[未就绪] .env 缺少：{', '.join(missing)}。请在 .env 里补齐后重跑。")
else:
    print(f"模型就绪：{settings.model_name} @ {settings.base_url}")
    print("（第 1、2、3.1、3.2、4、5 节需要真实模型；第 3.3 节是离线的。）")

## 1. 人工审核（Human-in-the-loop）

有些工具调用**有副作用**（删文件、转账、发消息），不该让模型自己决定就执行。
`HumanInTheLoopMiddleware` 在指定工具**真正执行之前**挂起整个图，等人类做决定。

四个决策（课案原文）：

| 决策 | 含义 |
|---|---|
| `approve` | 按原参数执行工具 |
| `reject` | 跳过本次调用，把拒绝信息回传给 Agent（**不是**终止整个 Agent） |
| `edit` | 修改工具名/参数后执行（本课 `09_人工审核_jxsd.py` 演示了它） |
| `respond` | 人类直接代替工具给出回复（本文件不演示） |

两条硬前提（缺一不可）：

1. **必须配 checkpointer** —— 中断状态靠检查点保存，没有它就没法「暂停后恢复」；
2. **恢复必须用同一个 `thread_id`** —— 否则找不到那份被冻结的检查点。

恢复执行的格式：`Command(resume={"decisions": [{"type": "approve"}]})`。

## 1.1 课案原版：最短的审核链路

原版只用两个工具（`delete_file` 危险 / `read_file` 安全），只对 `delete_file` 挂审核。
它走的是 LangGraph 的 **v1 返回结构**（`invoke` 直接返回状态字典，中断信息在
`result["__interrupt__"]` 里）—— 先看它，再看 1.2 的 v2 完整版。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


@tool
def delete_file(filename: str) -> str:
    """删除指定文件（危险操作，需要人工批准）"""
    return f"文件 {filename} 已删除"


@tool
def read_file(filename: str) -> str:
    """读取文件内容（安全操作，无需批准）"""
    return f"{filename} 的内容……"


agent = create_agent(
    model=llm,
    tools=[delete_file, read_file],
    system_prompt="你是文件管理助手。",
    checkpointer=MemorySaver(),  # 人工审核必须配 checkpointer
    middleware=[
        # 危险工具调用前挂起，等待人类决策
        HumanInTheLoopMiddleware(
            interrupt_on={
                "delete_file": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "删除文件属于危险操作，需要用户确认",
                },
            }
        ),
    ],
)

In [ ]:
config = {"configurable": {"thread_id": "hitl-1"}}

# 第一次执行：模型要删文件 → 挂起等待审核
result = agent.invoke(
    {"messages": [("user", "帮我把 tmp.txt 删掉")]}, config
)
print("挂起等待审核：", result.get("__interrupt__"))

# ---------- 分支 1：拒绝 ----------
result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "不许删！文件还要用"}]}),
    config,
)
print("拒绝后 AI：", result["messages"][-1].content)

# ---------- 分支 2：批准（换一个 thread_id 重新演示） ----------
config = {"configurable": {"thread_id": "hitl-2"}}
agent.invoke({"messages": [("user", "帮我把 tmp.txt 删掉")]}, config)
result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config,
)
print("批准后 AI：", result["messages"][-1].content)

### 预期输出

```text
挂起等待审核： [Interrupt(value={'action_requests': [{'name': 'delete_file', 'args': {'filename': 'tmp.txt'}, 'description': '删除文件属于危险操作，需要用户确认'}], 'review_configs': [{'action_name': 'delete_file', 'allowed_decisions': ['approve', 'reject']}]}, id='131c193e0e887839cc74711948c6beaa')]
拒绝后 AI： 收到，删除操作已被拒绝，`tmp.txt` 没有被删除，文件保持原样。
批准后 AI： 已完成，`tmp.txt` 已被删除。
```

> ⚠️ 模型措辞与检查点 `id` 每次运行都不同（AI 回复由模型生成、`id` 是随机检查点），
> 只有 `action_requests` 里的 `name` / `args` / `description` 稳定；上面的正文是本次实测值。

两个值得停一下看的点：

1. `__interrupt__` 里那个 `id='...'` 每次运行都不同，关键是 `action_requests` 里的
   `name` / `args` / `description` 稳定；
2. 拒绝后模型**自己改口**说「没删除」，批准后才真的删 —— 这就是「把决定权交回人」的效果。

## 1.2 完整版：三种决策 + 非交互脚本化

完整版（`09_人工审核_jxsd.py`）比原版多两样：

1. **`edit` 决策** —— `review()` 函数支持「多次修改后只提交一次决定」，是课案
   「允许多次修改」的实现；
2. **v2 返回结构** —— `invoke(..., version="v2")` 返回 `GraphOutput` 对象，
   中断信息在 `result.interrupts`、真正的状态字典在 `result.value`（老教程里没有这个）。

它原来的主循环是 `while True + input()` 死循环，在 notebook 无头执行时会 `EOFError`。
下面照模板第 6 节第 4 条，用 `scripted_input` 临时替换 `builtins.input`，让
**课案原样的 `review()` 函数**在非交互环境也能跑，并带兜底（审核轮数由模型决定，
写死答案长度会 `IndexError`）。**想真实交互，就把最后的调用换成 `demo_interactive()`。**

In [ ]:
import builtins
from copy import deepcopy                      # ← 课案原文：深拷贝待审核参数，避免改坏原对象
from pathlib import Path                       # ← 课案原文：写文件的工具需要

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from config import settings

# 课案写的是 ChatOpenAI(model=setting.MODEL_NAME, api_key=setting.API_KEY, base_url=setting.BASE_URL)
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

In [ ]:
# ---------- 1. 定义工具（课案原文） ----------
@tool
def write_file(content: str) -> str:
    """将内容写入 output.txt。"""
    # 课案原文：相对路径，落点是「运行时的当前工作目录」
    Path("output.txt").write_text(content, encoding="utf-8")
    return "已写入 output.txt"


# ---------- 2. 人工审核函数（课案原文，逐字保留） ----------
def review(action_request: dict) -> dict:
    """允许多次修改，最后只提交一次审核决定。"""
    action = {
        "name": action_request["name"],
        # ActionRequest 的参数字段名为 "args"（langchain 1.x，旧版为 "arguments"）
        "args": deepcopy(action_request["args"]),
    }
    edited = False

    while True:
        print(f"\n待审核：{action['name']} {action['args']}")
        choice = input("approve / reject / edit 补充内容: ").strip()

        # 三个分支对应课案说的三种决策，注意这里有一个容易忽略的设计：
        # 一旦用过 edit，后面的 approve 就不能再返回 {"type": "approve"}，
        # 否则前面改的参数会被丢掉 —— 所以 approve 分支要判断 edited 并返回 edited_action。
        if choice == "approve":
            return {"type": "edit", "edited_action": action} if edited else {"type": "approve"}
        if choice == "reject":
            # reject 只是「跳过本次工具调用 + 把拒绝信息回传给 Agent」，
            # 不是终止整个 Agent —— 模型收到拒绝后还会继续说话。
            return {"type": "reject"}
        if choice.startswith("edit"):
            # edited_action 的 args 就是 write_file 的入参，extra 直接拼到 content 后面；
            # 拼完 continue 回到循环顶部再问一次，这就是 docstring 里「允许多次修改」的实现。
            extra = choice[4:].strip()
            if extra:
                action["args"]["content"] += extra
                edited = True
            continue
        print("输入无效")

In [ ]:
# ---------- 3. 创建智能体（课案原文） ----------
model = llm

agent = create_agent(
    model=model,
    tools=[write_file],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": {
                    "allowed_decisions": ["approve", "reject", "edit"]
                }
            }
        )
    ],
    system_prompt="调用 write_file 前必须等待审核。被拒绝后不要再次调用。",
    checkpointer=InMemorySaver(),
)

# 课案原文的会话配置：审核必须靠 checkpointer + 同一个 thread_id 才能恢复
config = {"configurable": {"thread_id": "1"}}

In [ ]:
# ---------- 4. 一轮对话（课案 while True 循环体，抽成函数便于复现三种决策） ----------
def run_turn(user_text: str, turn_config: dict) -> None:
    """课案主循环体的一轮：invoke → 处理中断 → 恢复 → 打印 AI 回复。"""
    result = agent.invoke(
        {"messages": [("user", user_text)]},
        config=turn_config,
        version="v2",
    )

    # 只要有中断就继续处理：一次 invoke 可能挂起多轮（每轮审核一个工具调用）
    if not result.interrupts and not any(
        getattr(m, "tool_calls", None) for m in result.value["messages"]
    ):
        # 本机模型偶发「把工具调用说成一段文本」，此时不会产生中断；
        # 这不是审核机制失效，给一句中文提示，重跑一次即可。
        print("  ⚠ 本轮模型没有真正发起 write_file 工具调用（本机模型偶发行为），故无中断。")
        print("    模型输出：", str(result.value["messages"][-1].content)[:80])

    # 审核轮数是**模型**决定的（同一个提示词，模型可能比脚本预期多申请几轮），
    # 所以这里必须有轮数上限，避免模型反复申请审核时把演示卡死。
    rounds = 0
    while result.interrupts and rounds < MAX_REVIEW_ROUNDS:
        rounds += 1
        requests = result.interrupts[0].value["action_requests"]
        print(f"  [中断] 待审核动作 {len(requests)} 个，工具尚未执行")
        decisions = [review(request) for request in requests]
        print("  提交的决定：", decisions)
        result = agent.invoke(
            Command(resume={"decisions": decisions}),
            config=turn_config,
            version="v2",
        )
    if result.interrupts:
        print(f"  [提醒] 审核已达上限 {MAX_REVIEW_ROUNDS} 轮，本轮演示到此为止"
              "（模型仍在申请审核，不是代码卡死）")

    print("  AI:", result.value["messages"][-1].content)

In [ ]:
# ---------- 5. 适配：用预设答案替换 input()，让 review() 在非交互环境也能跑 ----------
# 审核轮数由模型决定，脚本化的答案序列只是「预期」，用完必须兜底 —— 否则
# 模型多申请一轮就会 `answers[0]` 抛 IndexError（本文件实测踩过：决策 B 脚本
# 给了 3 个答案，模型发起了第 4 轮审核）。兜底值取 approve：演示的目标是走完
# 「审核 → 执行」，不该卡在审核里。
MAX_REVIEW_ROUNDS = 6      # 单轮对话最多处理几次审核中断
DEFAULT_DECISION = "approve"


def scripted_input(answers: list[str], default: str = DEFAULT_DECISION):
    """把 builtins.input 临时换成「按顺序吐预设答案」，用于非交互演示。

    这样做的价值：跑的是**课案原样的 review() 函数**（同一条代码路径），
    而不是另写一段假逻辑，演示与真实交互的效果一致。

    预设答案用完后按 `default` 兜底，并在日志里说明 —— 让「脚本喂的答案不够」
    这件事可见，而不是静默改变演示语义。
    """
    answers = list(answers)

    def fake_input(prompt: str = "") -> str:
        if answers:
            answer = answers.pop(0)
        else:
            answer = default
            print(f"  （预设答案已用完，按兜底值处理：{answer}）")
        print(f"{prompt}{answer}")     # 回显，让日志看起来像真人输入的
        return answer

    return fake_input

In [ ]:
def demo_non_interactive() -> None:
    """非交互环境：三种决策各跑一轮（各自独立 thread_id，互不干扰）。"""
    print("检测到非交互环境（stdin 不是终端），自动演示三种审核决策。\n")
    scenarios = [
        # (说明, 用户输入, review() 里要喂的答案序列, thread_id)
        ("决策 A：approve —— 直接批准执行",
         "把「你好世界」写进 output.txt",
         ["approve"],
         "1-demo-approve"),
        ("决策 B：edit → approve —— 先改内容再批准（课案「允许多次修改」的效果）",
         "把「你好世界」写进 output.txt",
         ["edit （人工补充：请附带日期）", "edit 再补一句", "approve"],
         "1-demo-edit"),
        ("决策 C：reject —— 拒绝执行，拒绝信息回传给 Agent",
         "把「你好世界」写进 output.txt",
         ["reject"],
         "1-demo-reject"),
    ]

    for title, user_text, answers, thread_id in scenarios:
        # 三个场景各自用一个独立 thread_id：审核状态挂在 thread 上，
        # 共用 id 会让上一场景的检查点串进下一场景，三种决策就分不清了。
        print("=" * 60)
        print(title)
        print(f"用户: {user_text}")
        # 替换 builtins.input 而不是重写 review()：这样跑的还是课案原来的那条代码路径。
        original_input = builtins.input
        builtins.input = scripted_input(answers)    # 喂给 review() 的预设答案
        try:
            run_turn(user_text, {"configurable": {"thread_id": thread_id}})
        finally:
            builtins.input = original_input          # 一定要还原，避免影响后续代码

    # 审核通过时，工具真的执行了，磁盘上应当出现 output.txt
    target = Path("output.txt")
    print("\n" + "=" * 60)
    print(f"output.txt 是否生成：{target.exists()}"
          f"{'，内容=' + repr(target.read_text(encoding='utf-8')) if target.exists() else ''}")
    print("（approve / edit 两种情况会写文件；reject 不会。）")


def demo_interactive() -> bool:
    """交互环境：完全按课案原文跑，输入 exit / quit 结束。

    返回值：True = 正常结束；False = 读不到输入（无可用控制台）需走降级路径。
    注意：本机某些执行环境里 `sys.stdin.isatty()` 会返回 True，
    但真正 read 的时候立刻 EOF（比如被 IDE / 任务调度器接管的标准输入），
    所以这里还要再兜一层 EOFError —— 这正是规范第 5.4 条要防的坑。
    """
    print("交互模式：输入内容让 Agent 写文件，中途会停下来等你审核。")
    print("审核时可用：approve / reject / edit 补充内容；主循环输入 exit 退出。")
    while True:
        try:
            user_text = input("\n用户: ")
        except EOFError:
            return False                      # 读不到输入，交给调用方降级
        if user_text.strip().lower() in ("exit", "quit"):
            print("=== 对话结束 ===")
            break
        run_turn(user_text, config)
    return True

In [ ]:
# 课案是死循环 + input()，重定向输入时会 EOFError；
# 这里按规范做 isatty 判断（外加 EOF 兜底）：终端里走交互，否则走脚本化演示。
interactive_ok = False
if sys.stdin.isatty():
    interactive_ok = demo_interactive()
if not interactive_ok:
    if sys.stdin.isatty():
        print("[提示] 标准输入被判为终端但读不到内容（无可用控制台），改用脚本化演示。\n")
    demo_non_interactive()

### 预期输出

下面是**决策 B（edit → approve）**这一段节选（三种决策各自的 AI 回复因模型而异，
但「待审核 / 提交的决定 / 中断」这些结构性日志是稳定的）：

```text
决策 B：edit → approve —— 先改内容再批准（课案「允许多次修改」的效果）
用户: 把「你好世界」写进 output.txt
  [中断] 待审核动作 1 个，工具尚未执行

待审核：write_file {'content': '你好世界'}
approve / reject / edit 补充内容: edit （人工补充：请附带日期）

待审核：write_file {'content': '你好世界（人工补充：请附带日期）'}
approve / reject / edit 补充内容: edit 再补一句

待审核：write_file {'content': '你好世界（人工补充：请附带日期）再补一句'}
approve / reject / edit 补充内容: approve
  提交的决定： [{'type': 'edit', 'edited_action': {'name': 'write_file', 'args': {'content': '你好世界（人工补充：请附带日期）再补一句'}}}]
  AI: ...
```

> ⚠️ 本格输出含模型措辞，且**审核轮数由模型决定**：上面的正文只是本次实测值；脚本预设
> 答案用完时会走兜底（打印「预设答案已用完，按兜底值处理」），不同运行可能多审一轮或少审
> 一轮，所以 AI 回复与轮数每次运行都不同，只有「待审核 / 提交的决定」这些结构性日志稳定。

几个关键观察点：

1. `edit` 之后 `continue` 回循环顶部再问一次 —— 这就是「允许多次修改」，
   两次 edit 的内容被依次拼进 `content`；
2. 用过 edit 后再 `approve`，返回的是 `{'type': 'edit', 'edited_action': ...}`
   而不是 `{'type': 'approve'}` —— 否则前面改的参数会丢掉（见 `review()` 注释）；
3. `output.txt 是否生成` 那一行：approve / edit 两个场景会写文件，reject 不会；
4. 本次实测里决策 B 的模型多申请了一轮审核，脚本预设答案耗尽后走了 approve 兜底 ——
   这正是 `scripted_input` 里那个 `default` 兜底存在的意义（不会 `IndexError`）。

## 2. 中间件：钩子（Hooks）

人工审核只是中间件的一种。LangChain 的中间件本质上是一组**钩子**，让你在
智能体循环的各个环节插入自定义逻辑。钩子分两类（这是理解一切内置中间件的钥匙）：

| 维度 | 节点式（before/after_*） | 包裹式（wrap_*） |
|---|---|---|
| 在图里的位置 | 图上的**独立节点** | 不是节点，是「套在调用外面的一层壳」 |
| 你能拿到什么 | `state`（完整状态）+ `runtime` | `request`（即将发出的请求）+ `handler` |
| 能不能改状态 | 能，`return dict` 合并进 state | 能，改写 request 再交给 handler |
| 能不能不执行 | 只能 `jump_to` 跳走 | **能**：不调 handler（短路）或调多次（重试） |
| 典型用途 | 日志、裁剪历史、注入上下文 | 重试、缓存、限流、监控耗时 |

> 官方其实还有**第 7 个装饰器** `@dynamic_prompt`（动态系统提示词）——
> 它是 `wrap_model_call` 的便捷封装，课案 HTML 的六钩子表没列它，本节 2.2 补上。

## 2.1 课案原版：三个钩子做「日志 + 动态时间」

原版只演示三个钩子：`before_model`（记调用次数）、`after_model`（看最新消息）、
`dynamic_prompt`（每次注入当前时间）。

In [ ]:
from datetime import datetime

from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentState,
    before_model,
    after_model,
    dynamic_prompt,
)
from langchain.chat_models import init_chat_model
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ---------- 钩子 1：before_model（调模型前） ----------
def log_before(state: AgentState, runtime) -> dict | None:
    """记录调用次数；也可以在这里清洗/裁剪消息"""
    count = state.get("call_count", 0) + 1
    print(f"[before_model] 第 {count} 次调用模型")
    return {"call_count": count}


# ---------- 钩子 2：after_model（调模型后） ----------
def log_after(state: AgentState, runtime) -> dict | None:
    print(f"[after_model] 最新消息：{state['messages'][-1].content[:50]}")
    return None  # 不修改状态


# ---------- 钩子 3：dynamic_prompt（动态提示词） ----------
@dynamic_prompt
def inject_time(request):
    """每次调模型前动态生成系统提示词，注入当前时间"""
    return f"你是时间助手。当前时间：{datetime.now()}，回答要简短。"


# ---------- 组装中间件 ----------
agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        before_model(log_before),
        after_model(log_after),
        inject_time,
    ],
)

In [ ]:
result = agent.invoke({"messages": [("user", "现在是几点？")]})
print("AI：", result["messages"][-1].content)

### 预期输出

```text
[before_model] 第 1 次调用模型
[after_model] 最新消息：现在是 16:02。
AI： 现在是 16:02。
```

> ⚠️ 注入的是「当前时间」，所以这两行里的时分秒是时间戳、每次运行都不同；只有
> `[before_model] 第 1 次调用模型` 这一行稳定。上面的 16:02 是本次实测值。

`dynamic_prompt` 在模型调用**前**把当前时间塞进 system prompt —— 模型本身不知道
真实时间，它的回答引用的是注入进去的时间。这就是「动态」提示词的意义：信息每次现算，
而不是写死在 `create_agent(system_prompt=...)` 里。

## 2.2 完整版：六个钩子的触发顺序 + 第 7 个 dynamic_prompt

完整版把六个钩子全部定义出来，用一个「计算 3 + 5」的例子把**节点式与包裹式的顺序差异**
一次性看全。`add` 工具必然触发两轮「模型 → 工具」循环，正好把每个钩子都跑到位。

In [ ]:
from datetime import datetime
from typing import Any, Callable

from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentState,
    ModelRequest,
    ModelResponse,
    after_agent,
    after_model,
    before_agent,
    before_model,
    dynamic_prompt,
    wrap_model_call,
    wrap_tool_call,
)
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from config import settings


# ---------- 1. 定义工具（课案原文） ----------
@tool
def add(a: float, b: float) -> float:
    """返回 a + b 的结果"""
    return a + b


# ---------- 2. 定义 6 个钩子（课案原文） ----------
# 注意装饰器本身就把函数包装成了中间件对象，所以后面可以直接
# 把 on_before_agent 这类「函数名」塞进 middleware=[...]，不需要再调用一次。
#
# 签名统一是 (state, runtime)：
#   state   —— 当前的 Agent 状态（就是那个装着 messages 的字典）
#   runtime —— 运行时上下文（context、store、config 等，这里用不到）
# 返回值：dict 表示「把这几项合并进 state」，None 表示「什么都不改」。


@before_agent
def on_before_agent(state: AgentState, runtime) -> dict[str, Any] | None:
    print("[before_agent] Agent 启动")
    return None


# can_jump_to=["end"] 是「声明能力」：告诉框架这个钩子可能返回 {"jump_to": "end"}，
# 框架据此在图上预先连好一条通往结束节点的边。
# 不声明却返回 jump_to，运行时会直接报错 —— 这是新手最常踩的坑之一。
@before_model(can_jump_to=["end"])
def on_before_model(state: AgentState, runtime) -> dict[str, Any] | None:
    print(f"[before_model] 消息数: {len(state['messages'])}")
    # 课案的安全阀：消息超过 20 条就不再叫模型了，直接结束（防止上下文爆炸 / 死循环）。
    # 本文件只聊两句，走不到这个分支。
    if len(state["messages"]) > 20:
        return {"jump_to": "end"}
    return None


@after_model
def on_after_model(state: AgentState, runtime) -> dict[str, Any] | None:
    last = state["messages"][-1]
    # 模型这一轮是「要调工具」还是「给最终答复」，就靠 tool_calls 有没有内容来判断
    has_tools = "工具调用" if hasattr(last, "tool_calls") and last.tool_calls else "文本回复"
    print(f"[after_model] 模型返回: {has_tools}")
    # 为什么这个判断放在 after_model 里：它是「模型刚说完、还没轮到工具执行」的唯一时机，
    # 也是 Agent 循环里做分支决策（要不要继续、要不要打回）的落脚点。
    return None


@after_agent
def on_after_agent(state: AgentState, runtime) -> dict[str, Any] | None:
    print(f"[after_agent] Agent 结束，共 {len(state['messages'])} 条消息")
    return None


@wrap_model_call
def on_wrap_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    # 包裹式的核心是 handler：真正去调模型的是它。
    # 你可以在 handler 前后加逻辑，也可以「不调 handler 直接返回假响应」（短路），
    # 或者「调两次」（重试）——这正是 11_内置中间件_jxsd.py 里 ModelRetryMiddleware 的原理。
    #
    # 顺序上的关键区别（对照文件头第 2 条细节）：
    #   handler(request) 这一句「内部」才是模型的真实往返，
    #   所以 [wrap_model_call] → 与 ← 这两条日志，必然夹在 before_model 与 after_model 中间；
    #   而包裹式钩子本身**不会**在图的事件流里单独出现 —— 它不是节点。
    print("[wrap_model_call] → 调用模型")
    response = handler(request)
    print("[wrap_model_call] ← 模型返回")
    return response


@wrap_tool_call
def on_wrap_tool(request: Any, handler: Callable) -> Any:
    # request 是 ToolCallRequest，最关键的是 request.tool_call（含 name / args / id）
    print(f"[wrap_tool_call] 工具: {request.tool_call['name']}")
    response = handler(request)
    # response 是一条 ToolMessage，打印出来能看到 content / name / tool_call_id
    print(f"[wrap_tool_call] ← 工具返回: {response}")
    return response

In [ ]:
# ---------- 3. 创建 Agent 并运行（课案原文） ----------
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

agent = create_agent(
    model=llm,
    tools=[add],
    # 直接放装饰后的中间件对象；列表顺序 = 节点式钩子的执行顺序
    # 六个钩子的分工（对照文件头那张表看这里放的是什么）：
    #   on_before_agent / on_before_model / on_after_model / on_after_agent —— 节点式，
    #       在图上各占一个节点，按列表顺序依次跑；
    #   on_wrap_model / on_wrap_tool —— 包裹式，不是节点，
    #       分别「套」在模型调用和工具调用外面，所以放列表里的位置不影响它们的触发时机。
    middleware=[
        on_before_agent, on_before_model, on_after_model, on_after_agent,
        on_wrap_model, on_wrap_tool,
    ],
)

In [ ]:
# ---------- 补充：第 7 个装饰器 dynamic_prompt（课案精简版的「钩子 3」） ----------
# 课案 HTML 的六钩子表没列它，但课案精简版 10_中间件_钩子.py 把它当「钩子 3」用了。
# 官方实现（langchain 1.4.0）本质是 wrap_model_call 的便捷封装：每次调模型**前**
# 执行被装饰函数，把返回值设为本次请求的 system prompt。
# 注意签名跟六个钩子都不一样 —— 被装饰函数接收的是 request: ModelRequest
# （不是 (state, runtime)）：
#   request.state    —— 完整状态（想按消息数定制提示词就从这里取）
#   request.runtime  —— 运行时上下文（context / store / config）
# 返回值：str 或 SystemMessage。


@dynamic_prompt
def inject_time(request: ModelRequest) -> str:
    """每次调模型前动态生成系统提示词，注入当前时间（课案精简版「钩子 3」原文）。"""
    # 典型用途：把「模型天生不知道的事」实时喂进去 —— 当前时间、日期、用户身份。
    # 写在 create_agent(system_prompt=...) 里是静态的；这里每次现算。
    # （这句 print 是本文件加的观察点，课案原文没有）
    print(f"[dynamic_prompt] 注入系统提示词，当前时间 {datetime.now():%H:%M:%S}")
    return f"你是时间助手。当前时间：{datetime.now()}，回答要简短。"


# 单独再组一个 Agent：上面那个六个钩子的 Agent 是课案原文，保持原样不动；
# 这个只挂 dynamic_prompt，避免两套注入互相干扰、看不清谁在起作用。
agent_dynamic = create_agent(
    model=llm,
    tools=[],
    middleware=[inject_time],   # 装饰后即是中间件对象，直接放列表（同六个钩子）
)

In [ ]:
# 「计算 3 + 5」是课案原文的问法：它必定触发工具调用（第 1 轮），
# 再让模型把工具结果组织成话（第 2 轮），两轮循环才够把六个钩子的顺序演示完整。
print("===== 课案原文：六个钩子的触发顺序 =====")
result = agent.invoke({"messages": [{"role": "user", "content": "计算 3 + 5"}]})
# 对照文件头「课案给出的预期运行结果」看这行：整段日志的收尾就是它，
# 说明 after_agent（Agent 结束）确实发生在最终答复产生之后。
print(f"\n答案: {result['messages'][-1].content}")

# ---------- 4. 顺便看一眼状态里到底有什么 ----------
# 钩子日志看的是「过程」，这里看的是「结果」：
# 消息数应当是 4（user → ai(带 tool_calls) → tool → ai），
# 与文件头第 3 条细节「消息数从 1 → 3 → 4」逐条对得上。
print("\n===== 消息清单（对照上面的钩子日志） =====")
for index, msg in enumerate(result["messages"], start=1):
    # msg.type 就是 02_消息_jxsd.py 里那张表说的角色值（human / ai / tool）
    print(f"  [{index}] {msg.type:<7} {str(msg.content)[:60]}")

# ---------- 补充 Demo：dynamic_prompt（第 7 个装饰器，课案精简版的「钩子 3」） ----------
print("\n===== 补充：dynamic_prompt 动态注入系统提示词 =====")
# 观察点：[dynamic_prompt] 那行打印发生在模型调用之前；模型本身并不知道真实时间，
# 它的回答引用的正是注入进去的「当前时间」—— 这就是「动态」提示词的意义：
# 信息每次现算、随请求注入，而不是写死在 create_agent(system_prompt=...) 里。
result_dynamic = agent_dynamic.invoke(
    {"messages": [{"role": "user", "content": "现在是几点？"}]}
)
print("AI：", result_dynamic["messages"][-1].content)

### 预期输出

```text
===== 课案原文：六个钩子的触发顺序 =====
[before_agent] Agent 启动
[before_model] 消息数: 1
[wrap_model_call] → 调用模型
[wrap_model_call] ← 模型返回
[after_model] 模型返回: 工具调用
[wrap_tool_call] 工具: add
[wrap_tool_call] ← 工具返回: content='8.0' name='add' tool_call_id='call_...'
[before_model] 消息数: 3
[wrap_model_call] → 调用模型
[wrap_model_call] ← 模型返回
[after_model] 模型返回: 文本回复
[after_agent] Agent 结束，共 4 条消息

答案: 3 + 5 = **8**
```

> ⚠️ 六个钩子的「日志结构」稳定，但 `tool_call_id`（供应商随机生成）与「答案 / 消息清单」
> 里的模型措辞每次运行都不同；上面的正文是本次实测值。

三个值得注意的细节：

1. `before_agent` / `after_agent` 各只出现**一次**，`before_model` / `after_model`
   出现**两次** —— 因为「模型 → 工具」循环跑了 2 轮（第 1 轮决定调工具、第 2 轮组织语言）；
2. `[wrap_model_call] →` 与 `←` 夹在 `before_model` 与 `after_model` 之间 —— 它不是节点，
   而是「包住模型调用」的一层壳，所以不会单独出现在事件流里；
3. 消息数 1 → 3 → 4：+1 是带 tool_calls 的 AIMessage、+1 是 ToolMessage、最后一轮 +1 是纯文本 AI。
   （`tool_call_id` 是供应商生成的，每次运行都不同，所以写成 `call_...`。）

## 3. 内置中间件：开箱即用的「官方封装」

前面 2.2 手写了 6 个钩子，其实大部分横切需求官方已经封装好了。记住一句话：
**中间件不是新概念，它们全都是那六个钩子的官方封装**（比如
`ModelRetryMiddleware` = `wrap_model_call` + 重试循环，`TodoListMiddleware` =
`before_agent` 注入工具和提示词 + state 里的 `todos` 字段）。所以这节的重点不是
「会用」，而是「知道轮子已经造好了，别自己再写一遍」。

课案总表（7 个内置中间件）：

| 中间件 | 解决什么 | 触发后可见效果 | 关键参数 |
|---|---|---|---|
| `SummarizationMiddleware` | 历史太长撑爆上下文 | 旧历史被「摘要消息 + 最近消息」替代 | `model`、`trigger`、`keep` |
| `HumanInTheLoopMiddleware` | 工具有副作用 | 第一次调用返回 interrupt，等人工决定 | `interrupt_on`、`checkpointer` |
| `ModelCallLimitMiddleware` | Agent 死循环烧钱 | 到量直接结束并追加限制提示 | `run_limit`、`thread_limit` |
| `ModelRetryMiddleware` | 模型接口偶发超时 | 异常被吸收自动重试 | `max_retries`、`retry_on` |
| `ToolRetryMiddleware` | 外部服务偶发不可用 | 同一次工具调用连续重试 | `max_retries`、`tools` |
| `TodoListMiddleware` | 多步任务没规划 | 注入 `write_todos` 工具、`result["todos"]` 可读 | `system_prompt` |
| `ContextEditingMiddleware` | 工具返回太长占满上下文 | 旧工具结果变 `[cleared]` 占位符 | `edits` |

一句话选型：上下文太长 → Summarization / ContextEditing；怕烧钱 → ModelCallLimit；
外部服务不稳 → ModelRetry / ToolRetry；任务多步 → TodoList；工具有副作用 → HumanInTheLoop。

## 3.1 课案原版：七个中间件一次挂齐

原版只是把 7 个中间件**同时**挂在一个 Agent 上，让它们协同工作 —— 但一次挂齐
看不出每个中间件各自的效果。所以完整版（3.2）会把每个中间件拆成独立 Demo 逐个跑。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ContextEditingMiddleware,
    ModelCallLimitMiddleware,
    ModelRetryMiddleware,
    SummarizationMiddleware,
    TodoListMiddleware,
    ToolRetryMiddleware,
)
from langchain.chat_models import init_chat_model
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="你是简洁助手。",
    middleware=[
        # ① 历史消息超过一定 token 时，自动摘要成一条总结消息
        SummarizationMiddleware(
            model=llm,                # 用哪个模型来写摘要
            trigger=("tokens", 4000),  # 超过 4000 token 触发摘要
            keep=("messages", 4),     # 摘要时保留最近 4 条原文
        ),

        # ② 模型最多调 10 次，防止工具死循环烧钱（thread 级限流）
        ModelCallLimitMiddleware(thread_limit=10),

        # ③ 模型接口失败自动重试，最多 2 次，指数退避
        ModelRetryMiddleware(max_retries=2),

        # ④ 工具失败自动重试，最多 3 次
        ToolRetryMiddleware(max_retries=3),

        # ⑤ 任务清单：多步任务自动规划 TODO
        TodoListMiddleware(),

        # ⑥ 上下文清理：旧的超大工具结果替换为占位符，省 token
        ContextEditingMiddleware(),
    ],
)

In [ ]:
result = agent.invoke({"messages": [("user", "帮我总结一下 LangChain 的核心概念")]})
print("AI：", result["messages"][-1].content)

### 预期输出

```text
AI： # LangChain 核心概念总结（模型自由发挥的结构化总结，本次实测约 2000 字，此处省略）
```

> ⚠️ 这一格就是模型自由发挥的完整总结，措辞与长度每次运行都不同，无逐字比对意义；
> 稳定的是「它确实调用了模型并返回了总结」这件事。

原版这个 Agent 没有工具，所以只有 `SummarizationMiddleware` 的「检查是否该摘要」
会实际生效（这里历史很短、不触发），其余中间件都在「待命」状态 —— 这正是
3.2 要把每个中间件拆开单独触发的原因。

## 3.2 完整版：七个 Demo 逐个跑

每个 Demo 的固定结构 = 「它解决什么问题 → 关键参数什么意思 → 跑一遍看现象」。
七个 Demo 刻意顺序执行、互不 `try/except` 包住：每个都是独立的 agent + 独立的
thread_id，互不共享状态。

In [ ]:
import builtins
from pathlib import Path

from langchain.agents import create_agent
from langchain.agents.middleware import (
    ClearToolUsesEdit,
    ContextEditingMiddleware,
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ModelRequest,
    ModelRetryMiddleware,
    SummarizationMiddleware,
    TodoListMiddleware,
    ToolRetryMiddleware,
    wrap_model_call,
)
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from pydantic import PrivateAttr
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

In [ ]:
# ================================================================
# Demo 1：SummarizationMiddleware —— 历史太长就自动摘要
# ================================================================
# 课案原文：
#     trigger=("messages", 6)  # 达到 6 条消息就摘要
#     keep=("messages", 2)     # 将旧消息总结为一条摘要消息，同时保留最近两条原始消息
#
# 机制：每次调用模型**之前**检查一次。阈值一到，就把「老历史」交给摘要模型压成一条，
# 并给这条消息打上 lc_source="summarization" 标记（便于程序识别）。
# 副作用：触发摘要的那一轮会**多花一次模型调用**（摘要本身也要调模型）。
def demo_1_summarization() -> None:
    print("=" * 70)
    print("Demo 1：SummarizationMiddleware —— 自动摘要")
    print("=" * 70)

    agent = create_agent(
        model=llm,
        middleware=[
            SummarizationMiddleware(
                model=llm,                  # 用哪个模型写摘要（一般用便宜的小模型）
                trigger=("messages", 6),    # 达到 6 条消息就摘要
                keep=("messages", 2),       # 旧消息压成一条摘要，同时保留最近两条原文
                                            # 为什么还要 keep：摘要会丢细节，
                                            # 最近两条往往是当前话题的上下文，丢了模型就答偏。
            )
        ],
    )

    # 课案原文的 7 条历史：6 条「人机对话」+ 1 条新提问 = 触发阈值（>6）
    # 这个条数是**故意凑出来的**：trigger=("messages", 6) 是「达到 6 条就摘要」，
    # 而 invoke 时框架会先把新提问并进来，于是本轮请求实际带 7 条 → 刚好越线 → 必触发。
    # 想验证「不触发」的样子，把 history 删掉两条再跑即可。
    history = [
        HumanMessage(content="我计划去北京三天。"),
        AIMessage(content="可以安排故宫、长城和颐和园。"),
        HumanMessage(content="第一天想去故宫。"),
        AIMessage(content="建议提前预约上午场。"),
        HumanMessage(content="第二天去八达岭长城。"),
        AIMessage(content="可以乘坐高铁到八达岭长城站。"),
        HumanMessage(content="请根据前面的讨论给我一个简短建议。"),
    ]
    print(f"送入 {len(history)} 条历史消息（超过 trigger=6，本轮会先压缩再回答）")

    result = agent.invoke({"messages": history})
    # 关键现象：返回的消息数比送进去的还多。因为「压缩」不是删消息，而是**追加**一条摘要消息，
    # 只不过后续请求不会再带原始旧消息 —— 想验证这点，可以在下面循环里看那条 lc_source 标记。
    print(f"\n返回 {len(result['messages'])} 条消息（比送进去的还多，因为摘要那条是新增的）：")
    for index, value in enumerate(result["messages"], start=1):
        # 摘要消息带 lc_source 标记，据此把它和普通消息区分开
        # （additional_kwargs 是 LangChain 给消息挂「框架侧元数据」的地方，
        #   模型看不到它，程序可以靠它识别特殊消息）
        mark = " ← 摘要消息" if value.additional_kwargs.get("lc_source") == "summarization" else ""
        print(f"  [{index}] {type(value).__name__:<12}{mark}")
        print(f"       {str(value.content)[:110]}")

In [ ]:
# ================================================================
# Demo 2：HumanInTheLoopMiddleware —— 工具执行前审批
# ================================================================
# 课案这一段只写了「见上文人工审核」，完整实现见 09_人工审核_jxsd.py。
# 这里为了「每段都跑一遍」，做一个非交互的浓缩版：只演示 approve 一条路径，
# 三种决策（approve / reject / edit）的完整对照请看 09_人工审核_jxsd.py。
def demo_2_human_in_the_loop() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：HumanInTheLoopMiddleware —— 工具执行前审批（浓缩版，完整版见 09_人工审核_jxsd.py）")
    print("=" * 70)

    @tool
    def write_file(content: str) -> str:
        """将内容写入 output.txt。"""
        Path("output.txt").write_text(content, encoding="utf-8")
        return "已写入 output.txt"

    agent = create_agent(
        model=llm,
        tools=[write_file],
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={"write_file": {"allowed_decisions": ["approve", "reject"]}}
            )
        ],
        system_prompt="你是文件助手。用户要求写文件时，必须调用 write_file 工具；调用后等待人工审核。",
        # 审批必须能"暂停后恢复"，所以必须有 checkpointer
        checkpointer=MemorySaver(),
    )

    config = {"configurable": {"thread_id": "demo2-hitl"}}
    result = agent.invoke(
        {"messages": [("user", "请用 write_file 工具把「你好世界」写入 output.txt")]},
        config=config,
        version="v2",
    )

    if not result.interrupts:
        # 本机模型偶发「把 tool_call 说成文本」，此时不会产生中断 —— 给中文提示而不是报错
        # （判断依据：没有 interrupt 说明 HITL 中间件根本没被触发，而不是审核逻辑出错）
        print("  本轮模型没有发起工具调用（本机模型偶发行为），未产生中断。")
        print("  模型输出：", str(result.value["messages"][-1].content)[:80])
        print("  重新运行本 Demo 即可看到中断效果。")
        return

    # result.value 才是「真正的状态字典」：result 本身是 version="v2" 的 GraphOutput 包装，
    # value 里装的才是 messages —— 这是 LangChain/LangGraph 1.x 的新返回结构，老教程里没有。
    print("  第一次 invoke 返回中断：", result.interrupts[0].value["action_requests"])
    print("  工具尚未执行 → approval 决定：approve")
    # Command(resume=...) 就是「带着人工决定，从检查点原地复活」；
    # 必须复用同一个 config（thread_id），否则找不到那个被冻结的检查点。
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
        version="v2",
    )
    print("  恢复后 AI：", result.value["messages"][-1].content)
    print("  output.txt 是否生成：", Path("output.txt").exists())

In [ ]:
# ================================================================
# Demo 3：ModelCallLimitMiddleware —— 限制模型调用次数
# ================================================================
def demo_3_model_call_limit() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：ModelCallLimitMiddleware —— 限制模型调用次数")
    print("=" * 70)

    agent = create_agent(
        model=llm,
        middleware=[
            ModelCallLimitMiddleware(
                thread_limit=1     # 整个 thread 只允许调用 1 次模型
                # run_limit 是「单次 invoke 内」的限额，thread_limit 是「跨多轮对话」的累计限额
            )
        ],
        checkpointer=MemorySaver(),
    )
    config = {"configurable": {"thread_id": "limit-demo"}}
    # 两次 invoke 用的是**同一个** config —— 这正是 Demo 3 的全部要点：
    # thread_limit 统计的是这个 thread 的累计值，换个 thread_id 就又能调一次模型了。

    first = agent.invoke(
        {"messages": [{"role": "user", "content": "用一句话介绍 LangChain"}]},
        config=config,
    )
    print("第一次:", first["messages"][-1].content)

    # 同一个 thread 已调用过一次模型，第二次在调用模型前直接结束
    # （所以返回值里没有新的 AIMessage，只有中间件追加的限制提示 —— 这不算 traceback，是设计行为）
    second = agent.invoke(
        {"messages": [{"role": "user", "content": "再说一句"}]},
        config=config,
    )
    print("第二次:", second["messages"][-1].content)
    print("  ↑ 第二次没有真的问模型，而是被中间件拦下并追加了限制提示（exit_behavior 默认 'end'）")

In [ ]:
# ================================================================
# Demo 4：ModelRetryMiddleware —— 模型失败自动重试
# ================================================================
# 课案原文：定义一个「前两次必定超时」的模型子类，用来观察重试过程。
# 说明：_generate 是 LangChain 聊天模型的同步底层实现，invoke 最终会走到这里，
#       因此在它里面抛异常 = 模拟一次模型调用失败。
class FlakyChatOpenAI(ChatOpenAI):
    """前两次调用固定超时，第三次才请求真实模型。"""

    _attempts: int = PrivateAttr(default=0)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self._attempts += 1
        print(f"  模型尝试第 {self._attempts} 次")
        if self._attempts < 3:
            raise TimeoutError("模拟模型超时")
        return super()._generate(
            messages,
            stop=stop,
            run_manager=run_manager,
            **kwargs,
        )


def demo_4_model_retry() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：ModelRetryMiddleware —— 模型失败自动重试")
    print("=" * 70)

    model = FlakyChatOpenAI(
        model=settings.model_name,
        api_key=settings.api_key,
        base_url=settings.base_url,
    )
    agent = create_agent(
        model=model,
        middleware=[
            ModelRetryMiddleware(
                max_retries=2,          # 初始调用失败后，最多再重试几次
                retry_on=(TimeoutError,),  # 哪些异常需要重试
                initial_delay=0.0,      # 第一次重试前等待多少秒
                backoff_factor=0.0,     # 每次重试的等待时间增长倍数
            )
        ],
    )

    result = agent.invoke({"messages": [{"role": "user", "content": "回复：重试成功"}]})
    print("最终回答:", result["messages"][-1].content)
    # initial_delay / backoff_factor 都设成 0，是为了让教学时不必等退避时间；
    # 生产环境恰恰**不能**设 0 —— 对方服务正超时的时候立刻重试会把它打死（雪崩），
    # 标准做法是 initial_delay=1.0 起步、backoff_factor=2.0 翻倍、配 jitter 打散。
    print("  ↑ 第 1、2 次抛 TimeoutError 被中间件吞掉，第 3 次成功 → 总尝试 3 次 = 1 + max_retries")

In [ ]:
# ================================================================
# Demo 5：ToolRetryMiddleware —— 工具失败自动重试
# ================================================================
# 注意课案用的是模块级变量 + global：重试的是**同一次工具调用**，
# 所以计数器会跨重试累加（第 1、2 次抛异常，第 3 次返回真结果）。
attempts = 0


@tool
def get_weather(city: str) -> str:
    """查询城市天气。"""
    global attempts
    attempts += 1
    print(f"  工具尝试第 {attempts} 次")
    if attempts < 3:
        raise ConnectionError("模拟天气服务暂时不可用")
    return f"{city}：晴，26℃"


def demo_5_tool_retry() -> None:
    global attempts
    print("\n" + "=" * 70)
    print("Demo 5：ToolRetryMiddleware —— 工具失败自动重试")
    print("=" * 70)

    attempts = 0   # 重置计数器，保证本 Demo 可重复运行
    agent = create_agent(
        model=llm,
        tools=[get_weather],
        middleware=[
            ToolRetryMiddleware(
                max_retries=2,
                # tools 是**白名单**：不写就作用于所有工具。
                # 生产里通常要写 —— 只给「幂等的读操作」开重试，写操作不能碰（见文末踩坑 C）。
                tools=["get_weather"],        # 只对指定工具生效（按工具名过滤）
                retry_on=(ConnectionError,),  # 只重试这类异常
                initial_delay=0.0,
                backoff_factor=0.0,
            )
        ],
        system_prompt="查询天气时必须且只调用一次 get_weather 工具。",
    )

    result = agent.invoke({"messages": [{"role": "user", "content": "北京天气如何"}]})

    if attempts == 0:
        # attempts 还是 0 = 模型压根没发起工具调用，重试链一次都没进。
        # 这是本机模型的偶发行为，跟中间件无关；下面的 next() 也会因此抛 StopIteration，
        # 所以必须在这里提前 return（少了这句，脚本会在最后一行崩掉）。
        print("  本轮模型没有发起工具调用（本机模型偶发行为），重试逻辑未触发，请重新运行。")
        print("  模型输出：", str(result["messages"][-1].content)[:80])
        return

    # 从消息列表里挑出 get_weather 那条 ToolMessage：
    # 中间件重试了 3 次，但**只有成功那一次**会产生 ToolMessage —— 前两次的异常被吞掉了，
    # 消息列表里不会留下痕迹。这正是「包裹式钩子对模型透明」的直观证据。
    tool_result = next(
        message
        for message in result["messages"]
        if isinstance(message, ToolMessage) and message.name == "get_weather"
    )
    # 打印出来的是「第 3 次成功」的结果：前两次的 ConnectionError 在消息列表里查不到任何痕迹，
    # 因为包裹式钩子把异常吞在了模型看不到的那一层。
    print("工具结果:", tool_result.content)
    print("  ↑ 前两次 ConnectionError 被中间件吸收并重试，第 3 次成功，模型只看到成功结果")

In [ ]:
# ================================================================
# Demo 6：TodoListMiddleware —— 自动创建待办清单
# ================================================================
# 机制：这个中间件会「偷偷」给 Agent 注入一个 write_todos 工具 + 一段规划提示词，
#       模型调它就把结构化待办写进 state 的 todos 字段。
#       （所以模型眼里它和普通工具没区别，区别是它由中间件自带、不占你的 tools 列表。）
def demo_6_todo_list() -> None:
    print("\n" + "=" * 70)
    print("Demo 6：TodoListMiddleware —— 自动创建待办清单")
    print("=" * 70)

    agent = create_agent(
        model=llm,
        middleware=[TodoListMiddleware()]
        # 注意 tools=[] 是空的：write_todos 这个工具是中间件**自己注入**的，
        # 不需要（也不能）写进 tools 列表。这正是中间件的价值 —— 能力以插件形式加进来。
    )

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "请规划：调研竞品、设计接口、编写测试。",
                }
            ]
        }
    )

    todos = result.get("todos") or []
    # 用 .get 而不是 result["todos"]：todos 这个键只有在 TodoListMiddleware 启用、
    # 且模型真的调了 write_todos 之后才会出现在 state 里，直接下标会 KeyError。
    if not todos:
        print("  本轮模型没有调用 write_todos（本机模型偶发行为），请重新运行。")
        print("  模型输出：", str(result["messages"][-1].content)[:80])
        return

    for index, todo in enumerate(todos, start=1):
        # 课案只打印 content；这里额外把 status 也带上（pending / in_progress / completed），
        # 因为「状态」才是待办清单区别于普通文本的地方。
        # 本机模型可能把多行内容塞进一条，这里截断显示。
        content = str(todo["content"]).replace("\n", " ")[:70]
        print(f"  {index}. [{todo['status']}] {content}")

    print(f"\n  result['todos'] 共 {len(todos)} 条（这就是「结构化待办」——程序可直接读）")

In [ ]:
# ================================================================
# Demo 7：ContextEditingMiddleware —— 清理旧工具结果
# ================================================================
# 课案的注释：「第一个 wrap_model_call 是最外层，清理后再进入 show_model_context」。
# 下面用 @wrap_model_call 把「真正发给模型的消息」打印出来，让清理效果看得见：
#   - 清理只影响 **本次请求**（request.messages）；
#   - state["messages"]（完整状态）保持原样，历史不会被永久删除。
@wrap_model_call
def show_model_context(request: ModelRequest, handler):
    """课案注释里提到的 show_model_context：打印本次请求实际带上的上下文。

    它是一个**包裹式钩子**（不是节点）：被 @wrap_model_call 装饰后就成了中间件对象，
    可以直接放进 create_agent(middleware=[...]) —— 这正是 10 节讲的用法。
    这里把它和 ContextEditingMiddleware 串在一起，是为了让「清理」这件看不见的事变得可见。
    """
    print(f"  [show_model_context] 本次发给模型 {len(request.messages)} 条消息：")
    # 注意打印的是 request.messages，也就是「即将发给供应商接口的那一份」，
    # 和 state["messages"] 不是同一个对象 —— 这是本 Demo 的核心区别所在。
    for index, message in enumerate(request.messages, start=1):
        body = str(message.content).replace("\n", " ")[:46]
        print(f"      [{index}] {type(message).__name__:<12} {body!r}")
    # 洋葱模型里「往里走一层」：调 handler 才真正发起模型请求。
    # 想演示短路，把这一行换成 return AIMessage(...) 就行（不发请求直接给假答复）。
    return handler(request)


def demo_7_context_editing() -> None:
    print("\n" + "=" * 70)
    print("Demo 7：ContextEditingMiddleware —— 清理旧工具结果")
    print("=" * 70)

    agent = create_agent(
        model=llm,
        middleware=[
            # 第一个 wrap_model_call 是最外层，清理后再进入 show_model_context
            # 为什么顺序是这样：包裹式中间件是洋葱嵌套，列表**第一个在最外层**，
            # 所以「清理上下文」先执行，再轮到 show_model_context 打印 ——
            # 打印出来的就是清理后的结果，否则这个演示看不到任何效果。
            ContextEditingMiddleware(
                edits=[
                    ClearToolUsesEdit(
                        trigger=1,  # 当整个对话上下文超过约 1 个 token 时，开始清理旧工具结果。
                                    # —— 课案原文的极端值，等于「总是清理」，只为演示效果
                        keep=1,     # 最近 1 个工具结果永远保留，不清理
                                    # （保留最近的是为了不让模型丢失「刚查到的东西」）
                    )
                ]
            ),
            show_model_context,
        ],
    )

    # 手工拼一段「3 次工具调用 + 3 条工具结果」的历史：
    # 为什么是 3 次而不是 1 次 —— 因为 keep=1，只有存在**多条**旧工具结果时才看得出
    # 「前面的被清、最后一条留着」这个对比效果。全是手工构造的假数据，不会真的联网查天气。
    history = [
        HumanMessage(content="查询北京天气"),
        AIMessage(
            content="",
            tool_calls=[
                {"name": "get_weather", "args": {"city": "北京"}, "id": "call_1", "type": "tool_call"}
            ],
        ),
        ToolMessage(content="北京：晴，25℃", name="get_weather", tool_call_id="call_1"),
        HumanMessage(content="查询上海天气"),
        AIMessage(
            content="",
            tool_calls=[
                {"name": "get_weather", "args": {"city": "上海"}, "id": "call_2", "type": "tool_call"}
            ],
        ),
        ToolMessage(content="上海：小雨，22℃", name="get_weather", tool_call_id="call_2"),
        HumanMessage(content="查询深圳天气"),
        AIMessage(
            content="",
            tool_calls=[
                {"name": "get_weather", "args": {"city": "深圳"}, "id": "call_3", "type": "tool_call"}
            ],
        ),
        ToolMessage(content="深圳：多云，28℃", name="get_weather", tool_call_id="call_3"),
        HumanMessage(content="只根据已有结果总结三地天气，不要再次调用工具。"),
    ]

    result = agent.invoke({"messages": history})

    print("\n  ---- 完整状态 result['messages']（未被永久删除） ----")
    # 上下两段对照着看，Demo 7 的结论就出来了：
    #   上面 show_model_context 打印的「本次发给模型 N 条」里，旧工具结果已变成占位符；
    #   下面这份 result['messages']（也就是 state / checkpoint 里的原始消息）**一条没少**。
    # 一句话：ContextEditing 是「每次请求时的投影裁剪」，不是「删数据」。
    for index, value in enumerate(result["messages"], start=1):
        body = str(value.content).replace("\n", " ")[:46]
        print(f"      [{index}] {type(value).__name__:<12} {body!r}")

In [ ]:
# 7 个 Demo 刻意「顺序执行、互不 try/except 包住」：
# 每个 Demo 都是独立的 agent + 独立的 thread_id，互不共享状态；
# 只有 Demo 1 和 Demo 3 各自需要 checkpointer，且都在 Demo 内部自己建。
demo_1_summarization()
demo_2_human_in_the_loop()
demo_3_model_call_limit()
demo_4_model_retry()
demo_5_tool_retry()
demo_6_todo_list()
demo_7_context_editing()
print("\n全部 Demo 执行完毕。")

### 预期输出

七个 Demo 的输出较长，这里给**每个 Demo 的关键行**（完整版逐字见运行日志）：

```text
Demo 1：SummarizationMiddleware —— 自动摘要
送入 7 条历史消息（超过 trigger=6，本轮会先压缩再回答）
返回 4 条消息（历史被压缩成 1 条摘要 + 保留最近 2 条 + 新回答）：
  [1] HumanMessage  ← 摘要消息（lc_source="summarization"）
  [2] AIMessage       （保留的最近原文）
  [3] HumanMessage    （保留的最近原文）
  [4] AIMessage       （模型基于摘要给出的建议）

Demo 2：HumanInTheLoopMiddleware —— 工具执行前审批
  第一次 invoke 返回中断： [{'name': 'write_file', 'args': {'content': '你好世界'}, ...}]
  恢复后 AI： 已将「你好世界」写入 output.txt
  output.txt 是否生成： True

Demo 3：ModelCallLimitMiddleware —— 限制模型调用次数
第一次: ...（正常回答）
第二次: Model call limits exceeded: thread limit (1/1)（中间件追加的限制提示，没真问模型）

Demo 4：ModelRetryMiddleware —— 模型失败自动重试
  模型尝试第 1 次
  模型尝试第 2 次
  模型尝试第 3 次
最终回答: ...（模型基于成功结果作答）

Demo 5：ToolRetryMiddleware —— 工具失败自动重试
  工具尝试第 1 次
  工具尝试第 2 次
  工具尝试第 3 次
工具结果: 北京：晴，26℃

Demo 6：TodoListMiddleware —— 自动创建待办清单
  1. [in_progress] 调研竞品
  2. [pending] 设计接口
  3. [pending] 编写测试

Demo 7：ContextEditingMiddleware —— 清理旧工具结果
  [show_model_context] 本次发给模型 N 条消息：  （旧工具结果已变占位符）
  ---- 完整状态 result['messages'] ----  （原始消息一条没少）
```

> ⚠️ 除 Demo 4 / 7（自造异常与手工拼历史，无条件可复现）外，其余 Demo 的 AI 回复、消息
> 条数、待办内容都**由模型决定**，每次运行都不同；上面的正文是本次实测值，结构稳定。

注意 Demo 2 / 5 / 6 都带「本轮模型没有发起工具调用（本机模型偶发行为）」的兜底判断：
模型愿不愿意按剧本演是不确定的，那不是中间件失效，重跑一次即可。**无条件可复现**的是
Demo 4（自己造的 FlakyChatOpenAI 必抛超时）和 Demo 7（历史是手工拼的）。

## 3.3 官方补充：课案没讲的五个中间件（全离线）

课案只讲了 7 个中间件，官方文档还有 5 个课案没讲的（外加「课案 7 个中间件 vs 官方
真实签名」的核对结论）。本节**全离线可复现**：5 个 Demo 讲的全是中间件机制，模型只需要
按剧本「发起工具调用 / 给出文本」，于是全部改用**脚本模型**（继承 `ChatOpenAI`、覆写
`_generate` —— 与 3.2 Demo 4 的 `FlakyChatOpenAI` 是同一手法），**断网也能跑，输出逐字节稳定**。

| Demo | 中间件 | 解决什么 |
|---|---|---|
| 1 | `ToolErrorMiddleware` | 工具抛异常不再炸掉整个 Agent，转成 error 消息喂回模型 |
| 2 | `ModelFallbackMiddleware` | 主模型挂了自动切备用模型 |
| 3 | `ToolCallLimitMiddleware` | 限制**工具**调用次数（课案只讲了限**模型**的） |
| 4 | `PIIMiddleware` | 邮箱 / 密钥等敏感信息自动脱敏或直接拦截 |
| 5 | `LLMToolEmulator` | 不真跑工具，让 LLM 编一个结果（先把流程跑通） |

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    LLMToolEmulator,
    ModelFallbackMiddleware,
    PIIDetectionError,
    PIIMiddleware,
    ToolCallLimitMiddleware,
    ToolErrorMiddleware,
    ToolRetryMiddleware,
)
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import PrivateAttr


# ================================================================
# 脚本模型：本文件所有 Demo 的「模型」都由它扮演
# ================================================================
def ai_tool_call(name: str, args: dict, call_id: str) -> AIMessage:
    """构造剧本里的一行：「模型要求调用某个工具」。"""
    return AIMessage(
        content="",
        tool_calls=[{"name": name, "args": args, "id": call_id, "type": "tool_call"}],
    )


class ScriptedModel(ChatOpenAI):
    """按剧本依次吐消息的假模型。

    手法与 11_内置中间件_jxsd.py Demo 4 的 FlakyChatOpenAI 完全一样：
    继承 ChatOpenAI、只覆写 _generate —— bind_tools / 消息校验等框架方法全部
    沿用真实现，唯一被替换掉的是「真正发 HTTP 请求」那一步，所以断网也能跑。
    _received 记录每次模型**实际收到**的消息列表（Demo 4 靠它看见脱敏后的内容）。
    """

    _script: list = PrivateAttr(default_factory=list)
    _cursor: int = PrivateAttr(default=0)
    _received: list = PrivateAttr(default_factory=list)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self._received.append(list(messages))
        message = self._script[self._cursor]
        self._cursor += 1
        return ChatResult(generations=[ChatGeneration(message=message)])


def make_scripted(script: list) -> ScriptedModel:
    """造一个剧本模型。api_key / base_url 传假值即可：永远不会被真正用到。"""
    model = ScriptedModel(model="scripted", api_key="offline", base_url="http://localhost:9")
    model._script = script
    return model


def tool_outputs(result) -> list:
    """从结果里挑出所有 ToolMessage，打印用。"""
    return [m for m in result["messages"] if m.type == "tool"]

In [ ]:
# ================================================================
# Demo 1：ToolErrorMiddleware —— 工具炸了，Agent 不炸
# ================================================================
# 机制：它是一个 wrap_tool_call 包裹层。工具抛异常时，on_error(exc, request) 被调用；
# 返回字符串 → 转成一条 status="error" 的 ToolMessage 喂回模型（模型能据此改口/补救）；
# 返回 None   → 这个异常不归我管，原样往外抛（Agent 还是会崩）。
# 官方特意提醒：别把 exc 原文直接给模型 —— 异常文本可能带内部细节/敏感信息，
# 「模型能看到什么」由 on_error 说了算，这正是它比裸 try/except 安全的地方。
def demo_1_tool_error() -> None:
    print("=" * 70)
    print("Demo 1：ToolErrorMiddleware —— 工具抛异常不再炸掉整个 Agent")
    print("=" * 70)

    counter = {"attempts": 0}

    @tool
    def query_order(order_id: str) -> str:
        """查询订单状态（本 Demo 里必定抛异常）。"""
        counter["attempts"] += 1
        raise ValueError(f"订单 {order_id} 不存在")

    def on_error(exc: Exception, request) -> str | None:
        # request.tool_call 里有 name / args / id，需要的话可以按工具名分流处理
        if isinstance(exc, ValueError):
            return f"查询失败：{type(exc).__name__}。请检查单号后重试或换个单号。"
        return None

    # ---- Part A：只用 ToolErrorMiddleware（对照组）----
    # 没有它：工具一抛异常，整个 Agent 直接崩；
    # 有了它：异常变成一条 error ToolMessage，对话还能继续 —— 模型自己决定怎么收尾。
    model = make_scripted([
        ai_tool_call("query_order", {"order_id": "A001"}, "c1"),
        AIMessage(content="抱歉，订单查询失败了，请确认单号是否正确。"),
    ])
    agent = create_agent(model=model, tools=[query_order],
                         middleware=[ToolErrorMiddleware(on_error)])
    result = agent.invoke({"messages": [("user", "帮我查一下订单 A001")]})
    print(f"Part A：工具真实执行 {counter['attempts']} 次，Agent 全程没崩")
    for m in tool_outputs(result):
        print(f"  ToolMessage: status={m.status!r} content={str(m.content)[:52]!r}")
    print("  模型最终答复:", result["messages"][-1].content)

    # ---- Part B：官方组合拳 —— 先重试，耗尽后再转错误消息 ----
    # 关键是洋葱顺序（对照 10_中间件_钩子_jxsd.py 文件头那张两层表）：
    #   middleware 列表第一个 = 最外层；异常从工具往外冒，先穿过最内层。
    #   [ToolError(外), ToolRetry(内)]：Retry 贴着工具先重试 → 耗尽后
    #   on_failure="error" 把异常继续往外抛 → 外层 ToolError 接住转成错误消息。
    # ⚠️ 官方文档 built-in 页的**示例代码**把 ToolRetry 写在列表第一位（=外层），
    #    而它自己的说明文字却说 retry 应放 inner（=内层）—— 文字是对的，代码抄不得：
    #    顺序反了的话 ToolError 在内层先接住异常直接转消息，重试根本不会发生
    #    （实测：那种顺序下 attempts=1 就结束了；本 Part 的正确顺序下 attempts=2）。
    counter["attempts"] = 0
    model = make_scripted([
        ai_tool_call("query_order", {"order_id": "A001"}, "c1"),
        AIMessage(content="重试也没成功，建议您稍后再试或联系客服。"),
    ])
    agent = create_agent(
        model=model,
        tools=[query_order],
        middleware=[
            ToolErrorMiddleware(on_error),   # 外层：兜底，把异常转成模型能读懂的消息
            ToolRetryMiddleware(             # 内层：贴着工具，失败先重试
                max_retries=1,
                retry_on=(ValueError,),
                on_failure="error",   # 重试耗尽后把异常**往外抛**。
                                     # 默认 "continue" 会自己吞成错误消息，
                                     # 那样外层 ToolError 就永远没机会出手了 ——
                                     # 组合时必须显式改成 "error"，这是官方文档强调的点。
                initial_delay=0.0,    # 教学演示不等退避（生产建议 initial_delay=1.0）
                backoff_factor=0.0,
                tools=["query_order"],
            ),
        ],
    )
    result = agent.invoke({"messages": [("user", "再帮我查一次 A001")]})
    print(f"\nPart B：工具真实执行 {counter['attempts']} 次（1 次原始 + 1 次重试）")
    for m in tool_outputs(result):
        print(f"  ToolMessage: status={m.status!r} content={str(m.content)[:52]!r}")
    print("  ↑ 重试发生且耗尽后，异常穿过内层到达外层 ToolError —— 顺序对了才有这个效果")

In [ ]:
# ================================================================
# Demo 2：ModelFallbackMiddleware —— 主模型挂了自动切备用
# ================================================================
# 机制：主模型（create_agent 的 model=）失败后，按顺序尝试这里列的备用模型，
# 直到成功或全部耗尽。典型用法：供应商抖动降级、主贵备便宜的成本分级。
def demo_2_model_fallback() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：ModelFallbackMiddleware —— 主模型故障自动降级")
    print("=" * 70)

    class AlwaysDown(ChatOpenAI):
        """主模型：每次调用必挂，模拟供应商宕机 / 限流。"""

        def _generate(self, messages, stop=None, run_manager=None, **kwargs):
            raise RuntimeError("模拟供应商故障")

    # 备用模型是**位置参数**，可以一路写多个：ModelFallbackMiddleware(m1, m2, m3)
    # 依序尝试。传模型实例（而不是 "openai:xxx" 字符串）的好处：base_url / api_key
    # 完全由我们自己控制 —— 本项目统一从 config 读，见 01_模型_jxsd.py。
    backup = make_scripted([
        AIMessage(content="（这是备用模型的回答）主模型暂时不可用，已自动为您切换。"),
    ])
    agent = create_agent(
        model=AlwaysDown(model="primary", api_key="offline", base_url="http://localhost:9"),
        middleware=[ModelFallbackMiddleware(backup)],
        tools=[],
    )
    result = agent.invoke({"messages": [("user", "你好")]})
    print("主模型：每次调用必抛 RuntimeError")
    print("最终回答:", result["messages"][-1].content)
    print("  ↑ Agent 没有崩，回答来自备用模型 —— 用户侧无感")

In [ ]:
# ================================================================
# Demo 3：ToolCallLimitMiddleware —— 限制工具调用次数
# ================================================================
# 与课案 11 的 ModelCallLimitMiddleware 是一对兄弟，别记混：
#   ModelCallLimit 限**模型**调用次数，exit_behavior 默认 'end'（到量直接结束）；
#   ToolCallLimit   限**工具**调用次数，exit_behavior 默认 'continue'（到量拦截但继续）。
# exit_behavior 三种值：
#   'continue'（默认）—— 被拦的调用变成一条 error ToolMessage，Agent 继续走，
#                        模型看到「超限」的反馈后自己收尾；
#   'error'          —— 直接抛 ToolCallLimitExceededError，运行终止；
#   'end'            —— 直接结束整个运行，但**只支持限制单个工具**，
#                        其它工具还有 pending 调用时会 NotImplementedError。
# thread_limit（跨轮累计）需要配 checkpointer，run_limit（单轮）不需要 —— 同课案 Demo 3。
def demo_3_tool_call_limit() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：ToolCallLimitMiddleware —— 限制工具调用次数（默认 continue）")
    print("=" * 70)

    counter = {"calls": 0}

    @tool
    def get_weather(city: str) -> str:
        """查询城市天气。"""
        counter["calls"] += 1
        return f"{city}：晴"

    # 剧本：模型连着要两次天气 —— 第一次放行，第二次被限额中间件拦下
    model = make_scripted([
        ai_tool_call("get_weather", {"city": "北京"}, "c1"),
        ai_tool_call("get_weather", {"city": "上海"}, "c2"),
        AIMessage(content="北京查询成功；上海这次被限额拦住了，下轮再试。"),
    ])
    agent = create_agent(
        model=model,
        tools=[get_weather],
        middleware=[
            ToolCallLimitMiddleware(
                tool_name="get_weather",  # 只限这一个工具；不写 = 所有工具共用额度
                run_limit=1,              # 每次 invoke 最多真实执行 1 次
            ),
        ],
    )
    result = agent.invoke({"messages": [("user", "查北京和上海的天气")]})
    print(f"工具真实执行 {counter['calls']} 次（模型要了 2 次，第 2 次被拦）：")
    for m in tool_outputs(result):
        # 第 2 条 ToolMessage 的 status='error'，内容大意是
        # "Tool call limit exceeded. Do not call 'get_weather' again."
        print(f"  ToolMessage: status={m.status!r} content={str(m.content)[:60]!r}")
    print("  ↑ 被拦的调用没有执行工具函数，而是塞回一条超限提示 —— 这就是 'continue'")

In [ ]:
# ================================================================
# Demo 4：PIIMiddleware —— 敏感信息脱敏 / 拦截
# ================================================================
# 内置 5 种类型：email / credit_card / ip / mac_address / url；
# strategy 四种：'redact'（换成 [REDACTED_类型]）/ 'mask'（部分打码）/ 'hash'（确定性哈希）
#                / 'block'（检测到就抛 PIIDetectionError，运行中止）；
# detector 三种写法：正则字符串 / 编译后的正则 / 函数（返回 PIIMatch 列表，可做校验逻辑）；
# 作用面三个开关：apply_to_input（默认 True）/ apply_to_output（默认 False）
#                / apply_to_tool_results（默认 False）—— 想护住输出要自己打开。
# 实现层：输入侧替换的是**这次模型调用看到的那份消息**（输出侧脱敏走 after_model），
# 所以原文不会落到 checkpoint 里 —— 实测见 Part A 的对照（Demo 只证明"模型没看到 + 返回 state 是占位符"）。
def demo_4_pii() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：PIIMiddleware —— 邮箱 / 密钥自动脱敏，或直接拦截")
    print("=" * 70)

    # ---- Part A：redact —— 模型永远看不到原文 ----
    model = make_scripted([AIMessage(content="已收到您的信息。")])
    agent = create_agent(
        model=model,
        tools=[],
        middleware=[
            PIIMiddleware("email", strategy="redact"),          # 内置类型：邮箱
            PIIMiddleware(                                       # 自定义类型：API 密钥
                "api_key",
                detector=r"sk-[a-zA-Z0-9]{32}",                  # 直接给正则字符串就行
                strategy="redact",
            ),
        ],
    )
    result = agent.invoke({
        "messages": [("user", "我的邮箱 zhangsan@example.com，密钥 sk-" + "x" * 32)],
    })
    print("Part A（redact）")
    print("  模型实际收到:", str(model._received[0][-1].content))
    print("  state 里存的 :", str(result["messages"][0].content))
    print("  ↑ 两处都已是占位符：原文既没发给模型，也没落进状态（before_model 阶段就改掉了）")

    # ---- Part B：block —— 检测到就中止，模型一次都不会被调 ----
    model = make_scripted([AIMessage(content="不应被调用")])
    agent = create_agent(model=model, tools=[],
                         middleware=[PIIMiddleware("email", strategy="block")])
    try:
        agent.invoke({"messages": [("user", "邮箱 zhangsan@example.com")]})
        print("\nPart B（block）：未拦截（不符合预期！）")
    except PIIDetectionError as exc:
        print(f"\nPart B（block）：按预期抛出 PIIDetectionError：{exc}")
        print(f"  模型被调用次数：{len(model._received)}（拦在进模型之前，一次都没调）")

In [ ]:
# ================================================================
# Demo 5：LLMToolEmulator —— 不真跑工具，让 LLM 编一个结果
# ================================================================
# 用途：后端 API 还没开发好 / 第三方接口按次收费 / 压测 Agent 流程时，
# 让整条链路先跑通 —— 工具调用照常发生，但结果由另一个 LLM 现编。
# tools=None 时**所有**工具都被模拟；tools=["名字"] 只模拟指定的。
# ⚠️ 模拟结果的 ToolMessage status 是 'success'，模型分不出真假 —— 所以只用于测试。
def demo_5_llm_tool_emulator() -> None:
    print("\n" + "=" * 70)
    print("Demo 5：LLMToolEmulator —— 工具不真跑，由 LLM 编结果")
    print("=" * 70)

    counter = {"executed": 0}

    @tool
    def get_weather(city: str) -> str:
        """查询真实天气（本 Demo 中不应被执行）。"""
        counter["executed"] += 1
        return f"{city}：真实结果"

    # 生产中 model= 应传一个真实的小模型（便宜就行，它只负责「编得像」）；
    # 这里同样用脚本模型，保证本文件整体离线可复现。
    emu_model = make_scripted([AIMessage(content="晴，26℃（这是模拟模型编的结果）")])
    main_model = make_scripted([
        ai_tool_call("get_weather", {"city": "北京"}, "c1"),
        AIMessage(content="北京今天晴，26℃。"),
    ])
    agent = create_agent(
        model=main_model,
        tools=[get_weather],
        middleware=[LLMToolEmulator(tools=["get_weather"], model=emu_model)],
    )
    result = agent.invoke({"messages": [("user", "北京天气怎么样")]})
    print(f"真实工具执行次数：{counter['executed']}（应为 0 —— 调用被模拟器接管）")
    for m in tool_outputs(result):
        print(f"  ToolMessage: status={m.status!r} content={str(m.content)[:52]!r}")
    print(f"模拟模型被调用 {len(emu_model._received)} 次，它收到的指令开头是：")
    print("   ", str(emu_model._received[0][-1].content)[:110].replace("\n", " "))

In [ ]:
demo_1_tool_error()
demo_2_model_fallback()
demo_3_tool_call_limit()
demo_4_pii()
demo_5_llm_tool_emulator()
print("\n全部 Demo 执行完毕（0 次真实模型调用，离线可复现）。")

### 预期输出

```text
Demo 1：ToolErrorMiddleware —— 工具抛异常不再炸掉整个 Agent
Part A：工具真实执行 1 次，Agent 全程没崩
  ToolMessage: status='error' content='查询失败：ValueError。请检查单号后重试或换个单号。'
  模型最终答复: 抱歉，订单查询失败了，请确认单号是否正确。
Part B：工具真实执行 2 次（1 次原始 + 1 次重试）
  ↑ 重试发生且耗尽后，异常穿过内层到达外层 ToolError —— 顺序对了才有这个效果

Demo 2：ModelFallbackMiddleware —— 主模型故障自动降级
主模型：每次调用必抛 RuntimeError
最终回答: （这是备用模型的回答）主模型暂时不可用，已自动为您切换。
  ↑ Agent 没有崩，回答来自备用模型 —— 用户侧无感

Demo 3：ToolCallLimitMiddleware —— 限制工具调用次数（默认 continue）
工具真实执行 1 次（模型要了 2 次，第 2 次被拦）：
  ToolMessage: status='success' content='北京：晴'
  ToolMessage: status='error' content="Tool call limit exceeded. Do not call 'get_weather' again."
  ↑ 被拦的调用没有执行工具函数，而是塞回一条超限提示 —— 这就是 'continue'

Demo 4：PIIMiddleware —— 邮箱 / 密钥自动脱敏，或直接拦截
Part A（redact）
  模型实际收到: 我的邮箱 [REDACTED_EMAIL]，密钥 [REDACTED_API_KEY]
  state 里存的 : 我的邮箱 [REDACTED_EMAIL]，密钥 [REDACTED_API_KEY]
Part B（block）：按预期抛出 PIIDetectionError：Detected 1 instance(s) of email in text content
  模型被调用次数：0（拦在进模型之前，一次都没调）

Demo 5：LLMToolEmulator —— 工具不真跑，由 LLM 编结果
真实工具执行次数：0（应为 0 —— 调用被模拟器接管）
  ToolMessage: status='success' content='晴，26℃（这是模拟模型编的结果）'
模拟模型被调用 1 次，它收到的指令开头是：
   You are emulating a tool call for testing purposes.  Tool: get_weather Description: 查询真实天气（本 Demo 中不应被执行）。 Arg

全部 Demo 执行完毕（0 次真实模型调用，离线可复现）。
```

这一节是全 notebook 里唯一**输出逐字节稳定**的段落：脚本模型按固定剧本吐消息，
不依赖真实模型，所以「预期输出」可以和实跑完全一致（上面每一行都会在实跑输出里逐字出现）。

## 4. 上下文工程总纲（官方补充）

官方文档 `context-engineering` 的开篇把话挑明：Agent 失败十有八九不是模型不行，
而是**该给的上下文没给**。它把「上下文」切成**三类控制 × 三种数据源**：

| 控制类型 | 管什么 | 瞬时/持久 |
|---|---|---|
| Model Context | 单次模型调用看到什么（提示词/消息/工具/模型/响应格式） | 瞬时 |
| Tool Context | 工具能读到与写出什么（state/store/context） | 持久 |
| Life-cycle Context | 模型调用与工具调用**之间**发生什么（摘要、护栏、日志） | 持久 |

| 数据源 | 别名 | 作用域 |
|---|---|---|
| Runtime Context | 静态配置 | 会话级 |
| State | 短期记忆 | 会话级 |
| Store | 长期记忆 | 跨会话 |

机制：**所有上下文控制最终都落在中间件钩子上**（第 2 节已打底）。本节只补课案与
既有补充篇还没覆盖的 4 格：动态裁剪工具集（Demo 1）、动态切换输出格式（Demo 2）、
工具读三种数据源（Demo 3）、最小可观测（Demo 4）。

In [ ]:
import time
from typing import Union

from langchain.agents import create_agent
from langchain.agents.middleware import after_model, before_model, wrap_model_call
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.tools import ToolRuntime, tool
from langgraph.graph import MessagesState
from langgraph.store.memory import InMemoryStore
from pydantic import BaseModel, Field

from config import settings

model = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def final_text(result: dict) -> str:
    return str(result["messages"][-1].content)

In [ ]:
# ================================================================
# Demo 1：Model Context × Tools —— 按调用方动态裁剪工具集
# ================================================================
# 官方 Model Context 一节把 "Tools" 单列为一项可控内容：**给模型看哪些工具**是可调的。
# 现实需求：同一个 agent 服务不同套餐的用户 —— 免费用户不该看到「删除」这类危险工具。
# 做法：在 wrap_model_call 里用 request.override(tools=...) 换一份工具列表。
# 好处：工具根本没进提示词，模型**不可能**调用它（比"调了再拒绝"更安全）。
class CallerInfo(BaseModel):
    """本次调用的上下文：谁在调、什么套餐。

    注意：不管用 dataclass 还是 pydantic 模型，只要 invoke 时传了 context，
    langgraph 序列化都会打 `PydanticSerializationUnexpectedValue` 告警
    （实测两种写法都有，属无害噪音 —— 16_测试与护栏_官方补充.py 里记录过同一现象）。
    """

    user_id: str
    # 默认给最低权限（fail-closed）：拿不到身份时不该默认拿到高权限
    plan: str = "free"     # "free" / "pro"

# 记录模型每次调用实际看到的工具名，用来证明裁剪生效
seen_tools: list[list[str]] = []


@wrap_model_call
def limit_tools_by_plan(request, handler):
    """按套餐裁剪工具集：免费用户看不到 delete_record。"""
    # 取不到 context（未注入）或没声明 plan 时**按最低权限处理** ——
    # 安全相关的默认值必须 fail-closed：宁可少给工具，也不能默认放行。
    plan = getattr(request.runtime.context, "plan", None) or "free"
    allowed = [t for t in request.tools if not (plan == "free" and t.name == "delete_record")]
    seen_tools.append([t.name for t in allowed])
    return handler(request.override(tools=allowed))


@tool
def search_records(keyword: str) -> str:
    """按关键词查询记录。"""
    return f"查到 2 条包含 {keyword!r} 的记录"


@tool
def delete_record(record_id: str) -> str:
    """删除一条记录（危险操作）。"""
    return f"已删除 {record_id}"


def demo_1_dynamic_tools() -> None:
    print("=" * 70)
    print("Demo 1：Model Context × Tools —— 按套餐动态裁剪工具集")
    print("=" * 70)

    agent = create_agent(
        model=model,
        tools=[search_records, delete_record],
        middleware=[limit_tools_by_plan],
        context_schema=CallerInfo,
    )

    for plan in ("free", "pro"):
        seen_tools.clear()
        result = agent.invoke(
            {"messages": [{"role": "user", "content": "帮我查一下 keyword=订单"}]},
            context=CallerInfo(user_id=f"u-{plan}", plan=plan),
        )
        print(f"  {plan:<5} 用户 → 模型看到的工具：{seen_tools[-1] if seen_tools else '（没记录）'}")
        print(f"        回答：{final_text(result)[:70]}")

    print(
        "  ↑ 免费用户的提示词里**压根没有** delete_record —— 这是「工具级上下文控制」，\n"
        "    比「让模型别调、调了再拒绝」更可靠：不可见即不可调用。\n"
        "    override 还能换 model / messages / tool_choice / response_format（见 Demo 2）。"
    )

In [ ]:
# ================================================================
# Demo 2：Model Context × Response Format —— 按状态动态切换输出格式
# ================================================================
# 课案 08_结构化输出 讲的是「固定的 schema」；官方这里强调的是**动态**：
# 同一个 agent，在处理不同类型请求时用不同的输出格式（甚至有时不用结构化输出）。
# 场景：首轮把用户需求整理成结构化任务单，之后自由对话即可。
#
# ⚠️ 官方文档示例是「中间件里 override 成创建时没声明过的 schema」：
#       request = request.override(response_format=SimpleResponse)
#    但 langchain 1.x **不允许**这样做 —— 实测直接抛 ValueError：
#       ToolStrategy specifies tool 'TaskTicket' which wasn't declared
#       in the original response format when creating the agent.
#    所以**正确姿势**是：创建 agent 时用 ToolStrategy(Union[...]) 把候选 schema
#    一次声明齐，中间件只做「收窄到某一个」或「关掉」。
class TaskTicket(BaseModel):
    """首轮用的结构化任务单（创建 agent 时就声明，供中间件选用）。"""

    goal: str = Field(description="用户目标，一句话")
    steps: list[str] = Field(description="拆解出的 3 个步骤")


class FreeReply(BaseModel):
    """后续轮次用的自由回复（同样在创建时声明）。"""

    reply: str = Field(description="给用户的自然语言回复")


format_log: list[str] = []


@wrap_model_call
def dynamic_response_format(request, handler):
    """按消息数切换输出格式：首轮收窄成任务单，之后收窄成自由回复。"""
    if len(request.messages) <= 1:
        format_log.append("收窄为 TaskTicket（结构化任务单）")
        # 只收窄到「创建时声明过的子集」—— 这是框架允许的用法
        return handler(request.override(response_format=ToolStrategy(TaskTicket)))
    format_log.append("收窄为 FreeReply（自由回复）")
    return handler(request.override(response_format=ToolStrategy(FreeReply)))


def demo_2_dynamic_format() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：Model Context × Response Format —— 按状态切换输出格式")
    print("=" * 70)

    agent = create_agent(
        model=model,
        tools=[],
        middleware=[dynamic_response_format],
        # 候选 schema 必须在创建 agent 时一次声明齐（Union 即可）
        response_format=ToolStrategy(Union[TaskTicket, FreeReply]),
    )

    # ---- 第 1 轮：消息只有 1 条 → 中间件收窄成 TaskTicket，要结构化任务单 ----
    try:
        first = agent.invoke({
            "messages": [{"role": "user", "content": "我想给团队做个每周自动汇总的机器人"}],
        })
    except Exception as exc:
        # 结构化输出要靠端点能力：原生 json_schema，或「强制 tool_choice」。
        # 本机 .env 若指向 DeepSeek（deepseek-flash / deepseek-v4-pro 都是思考模型），
        # 两条路都不支持，会在这里抛 400 —— 兜成中文提示，不崩也不假装成功。
        print(f"  [跳过] 本端点不支持结构化输出：{type(exc).__name__}")
        print(f"         {str(exc)[:160]}")
        print("         把 .env 的 API_KEY/BASE_URL/MODEL_NAME 切回支持 json_schema 的端点，")
        print("         或删掉 response_format= 只演示工具裁剪即可跑通（见文末实测结论第 5 条）。")
        return

    structured = first.get("structured_response")
    print(f"  第 1 轮决策：{format_log[-1]}")
    if structured is not None:
        print(f"    结构化类型：{type(structured).__name__}")
        print(f"      目标：{getattr(structured, 'goal', None)}")
        print(f"      步骤：{getattr(structured, 'steps', None)}")
    else:
        print(f"    没拿到结构化结果，文本：{final_text(first)[:80]}")

    # ---- 第 2 轮：把上一轮消息带进来 → 消息数 > 1，中间件换成自由回复 ----
    second = agent.invoke({
        "messages": [*first["messages"],
                     {"role": "user", "content": "先按这个思路，帮我列一句对外介绍语"}],
    })
    structured2 = second.get("structured_response")
    print(f"  第 2 轮决策：{format_log[-1]}")
    if structured2 is not None:
        print(f"    结构化类型：{type(structured2).__name__}")
        print(f"      回复：{getattr(structured2, 'reply', None)}")
    else:
        print(f"    没拿到结构化结果，文本：{final_text(second)[:80]}")

    print(
        "  ↑ 同一个 agent，**按上下文决定这轮用哪个 schema** —— 这就是 Model Context 的\n"
        "    「瞬时」性质：改的是这一次模型调用看到的东西，state 里存的仍是普通消息。\n"
        "    实用场景：首轮抽任务单、后续自由对话；或简单问题跳过结构化输出省 token。"
    )

In [ ]:
# ================================================================
# Demo 3：Tool Context × 三种数据源 —— 工具能读到什么、能写到哪里
# ================================================================
# 官方 Tool Context 的说法：工具能读写的三处 —— Runtime Context（静态配置）、
# State（短期记忆）、Store（长期记忆）。`ToolRuntime` 对象正好把三样都带齐了
#   实测字段：state / context / config / stream_writer / tool_call_id / store /
#            tools / execution_info / server_info
# 本 Demo 演示：一次工具调用里**同时读三种数据源**，并把结果写进长期记忆（Store）。
class TicketState(MessagesState):
    """自定义状态：**必须继承 MessagesState**（它带了 add_messages reducer）。

    踩坑记录：一开始这里写成裸 TypedDict 并自己声明 `messages: list`，
    结果丢掉了 reducer —— 消息被后续节点覆盖，agent 的 while 循环判不出终止条件，
    整个运行**无限调用模型**（实测卡死 10 分钟没有任何输出）。
    正确姿势就是继承 MessagesState，再加自己的字段。
    """

    ticket_no: str          # 自定义状态字段：票据号


@tool
def inspect_context(runtime: ToolRuntime) -> str:
    """把工具能看到的三类上下文都读出来（演示 Tool Context 的数据源）。"""
    # ① Runtime Context：本次运行的静态配置（谁在调、什么环境）
    user = getattr(runtime.context, "user_id", "（未注入）")
    # ② State：本次会话的短期记忆（含自定义字段）
    ticket = runtime.state.get("ticket_no", "（state 里没有 ticket_no）")
    message_count = len(runtime.state.get("messages", []))
    # ③ Store：跨会话的长期记忆（读写都在这里）
    namespace = ("users", user)
    previous = runtime.store.get(namespace, "last_seen") if runtime.store else None
    runtime.store.put(namespace, "last_seen", {"note": "刚刚查询过上下文"}) if runtime.store else None
    return (
        f"runtime.context.user_id={user}；"
        f"state.ticket_no={ticket}，state 里已有 {message_count} 条消息；"
        f"store 里的历史记录={previous.value if previous else '（首次）'}"
    )


def demo_3_tool_context_sources() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：Tool Context —— 工具同时读 Runtime Context / State / Store")
    print("=" * 70)

    store = InMemoryStore()
    agent = create_agent(
        model=model,
        tools=[inspect_context],
        store=store,                       # 长期记忆要显式传进去
        state_schema=TicketState,          # 自定义状态字段：票据号
        context_schema=CallerInfo,
    )
    config = {"configurable": {"thread_id": "tool-context-demo"}}

    result = agent.invoke(
        {
            "messages": [{"role": "user", "content": "先看看你现在能拿到哪些上下文信息"}],
            "ticket_no": "T-2026-0917",
        },
        config,
        context=CallerInfo(user_id="u-1001", plan="pro"),
    )
    for message in result["messages"]:
        if message.type == "tool":
            print(f"  工具返回：{message.content}")
    print(f"  state 里的 ticket_no：{result.get('ticket_no')}")

    # 再跑一次：这次 Store 里已经有上次写进去的记录了（跨会话/跨轮次）
    result2 = agent.invoke(
        {"messages": [{"role": "user", "content": "再查一次上下文"}]},
        {"configurable": {"thread_id": "tool-context-demo-2"}},   # 换会话，但 Store 共享
        context=CallerInfo(user_id="u-1001", plan="pro"),
    )
    for message in result2["messages"]:
        if message.type == "tool":
            print(f"  第二次（换了 thread_id）工具返回：{message.content}")
    print(
        "  ↑ 一次工具调用把三种数据源都用上了：\n"
        "    · runtime.context —— 静态配置（会话级，由 invoke 注入）；\n"
        "    · runtime.state  —— 短期记忆（会话级，含自定义字段 ticket_no）；\n"
        "    · runtime.store  —— 长期记忆（跨会话：第二次换了 thread_id 仍读得到）。\n"
        "    这三者的边界画清楚，『该把数据放哪』就不再是拍脑袋。"
    )

In [ ]:
# ================================================================
# Demo 4：Life-cycle Context —— 模型调用之间发生了什么（最小可观测实现）
# ================================================================
# 官方的第三类上下文：模型调用与工具调用**之间**发生的事情 —— 摘要、护栏、日志…
# 课案与补充篇已有「摘要」（12_记忆 Demo 1）和「护栏」（16_测试与护栏 Demo 3），
# 这里补最小可观测实现：用 before_model / after_model 钩子统计每次调用的耗时与
# 该轮产生的工具调用数 —— 生产排障（"为什么这次回答特别慢"）的第一手数据。
lifecycle_stats: dict[str, float] = {"calls": 0, "elapsed": 0.0, "tool_calls": 0}


@before_model
def start_timer(state, runtime):
    lifecycle_stats["_t0"] = time.time()


@after_model
def record_after_model(state, runtime):
    lifecycle_stats["calls"] += 1
    lifecycle_stats["elapsed"] += time.time() - lifecycle_stats.get("_t0", time.time())
    # 这一轮模型消息里请求了几个工具调用（按消息结构直接数，不额外调模型）
    last = state["messages"][-1] if state.get("messages") else None
    lifecycle_stats["tool_calls"] += len(getattr(last, "tool_calls", None) or [])


def demo_4_lifecycle_observability() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：Life-cycle Context —— 用钩子做最小可观测")
    print("=" * 70)

    agent = create_agent(
        model=model,
        tools=[search_records],
        middleware=[start_timer, record_after_model],
    )
    result = agent.invoke({"messages": [{"role": "user", "content": "查一下 keyword=发票 的记录"}]})
    print(f"  回答：{final_text(result)[:80]}")
    print(f"  本次运行统计：模型调用 {int(lifecycle_stats['calls'])} 次，"
          f"累计耗时 {lifecycle_stats['elapsed']:.2f} 秒，"
          f"模型请求的工具调用 {int(lifecycle_stats['tool_calls'])} 个")
    print(
        "  ↑ 这就是官方说的 Life-cycle Context：钩子挂在模型/工具调用之间，\n"
        "    改的是**持久**的东西（日志、指标、状态），而不是单次提示词。\n"
        "    生产里把这两个钩子换成上报 Langfuse/OpenTelemetry 就是全链路追踪的起点。"
    )

In [ ]:
demo_1_dynamic_tools()
demo_2_dynamic_format()
demo_3_tool_context_sources()
demo_4_lifecycle_observability()
print("\n全部 Demo 执行完毕。")

### 预期输出

```text
Demo 1：Model Context × Tools —— 按套餐动态裁剪工具集
  free  用户 → 模型看到的工具：['search_records']
        回答：查到 2 条包含 '订单' 的记录
  pro   用户 → 模型看到的工具：['search_records', 'delete_record']
        回答：查到 2 条包含 '订单' 的记录
  ↑ 免费用户的提示词里压根没有 delete_record —— 不可见即不可调用

Demo 2：Model Context × Response Format —— 按状态切换输出格式
  [跳过] 本端点不支持结构化输出：OpenAIInvalidRequestError
         （当前 deepseek-flash 是思考模型，不支持 json_schema / 强制 tool_choice）

Demo 3：Tool Context —— 工具同时读 Runtime Context / State / Store
  工具返回：runtime.context.user_id=u-1001；state.ticket_no=T-2026-0917，...
  第二次（换了 thread_id）工具返回：...store 里的历史记录={'note': '刚刚查询过上下文'}

Demo 4：Life-cycle Context —— 用钩子做最小可观测
  本次运行统计：模型调用 N 次，累计耗时 X.XX 秒，模型请求的工具调用 M 个
```

> ⚠️ Demo 1/3/4 的 AI 回复、Demo 2 的报错细节、Demo 4 的耗时统计都**由模型/端点决定**、
> 每次运行都不同；稳定的是 Demo 1 的「模型看到的工具」列表与 Demo 3 的工具返回结构。
> 上面的正文是本次实测值。

**Demo 2 的降级说明**：动态 `response_format` 要真跑通，同时受两个条件限制——
① 框架允许（只能收窄到创建时声明过的 schema，本节已照正确姿势写）；② 端点支持。
当前 `deepseek-flash` 是思考模型，两条结构化输出路径（原生 json_schema / 强制 tool_choice）
都不支持 → 打印 `[跳过]`。切到支持 json_schema 的端点即可看到「首轮任务单、后续自由回复」的切换。

## 5. 自己组装 harness（官方补充）

官方 `deep-agent-from-scratch` 一篇的核心主张：`create_agent` 与 `create_deep_agent`
都能细粒度控制工具与记忆，区别只是 **Deep Agents 默认就把常用能力装好了**。如果默认
harness 不合适，就从 `create_agent` 起步，一件一件装出来，看清每件各自解决什么问题。

官方把组装拆成五步，每一步针对一个具体痛点：

| 步骤 | 不加会怎样 | 加了什么 |
|---|---|---|
| 0 最小 agent | 只有「模型 + 循环」 | （基线，没有 harness） |
| 1 沙箱 + 文件系统 | agent 读不到 CSV、跑不了 Python | 隔离后端 + 文件/执行工具 |
| 2 摘要压缩 | 长会话撞上下文上限 | 自动压缩历史 |
| 3 技能（Skills） | 领域规则把 system prompt 撑爆 | 按需加载（渐进披露） |
| 4 子代理 | 图表迭代挤占主线程 | 隔离的 worker + 并行委派 |

> 本文件与官方教程有两处不同（都为了让它在**本机**可跑）：
> A. 官方用 `LangSmithSandbox`（要 `LANGSMITH_API_KEY`），本仓库不用 LangSmith，
>    改用 `FilesystemBackend + 临时目录`（隔离程度弱于沙箱，注释里已标明）；
> B. 官方主题是「数据分析 agent」，这里把数据换成更小的示例，把篇幅留给
>    「每件中间件各自解决了什么」。

In [ ]:
import tempfile
from pathlib import Path

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

from config import settings

model = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


def text_of(message) -> str:
    """取消息文本：content 可能是字符串，也可能是内容块列表（LangChain 1.x）。"""
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(block.get("text", ""))
            else:
                parts.append(str(block))
        return "".join(parts)
    return str(content)


def tool_names_of(middlewares) -> list[str]:
    """列出这组中间件**贡献**了哪些工具（中间件自己就带着工具清单）。

    实测：编译后的 agent 图里 tools 节点是 PregelNode，拿不到工具名；
    而每个中间件实例上的 `.tools` 就是它注册的工具（FilesystemMiddleware
    实测给出 ls / read_file / write_file / edit_file / delete / glob / grep / execute）。
    """
    names: list[str] = []
    for middleware in middlewares:
        for item in getattr(middleware, "tools", None) or []:
            names.append(getattr(item, "name", str(item)))
    return sorted(set(names))


def ask(agent, question: str, config: dict | None = None) -> tuple[str, list[str]]:
    """跑一次 agent，返回（最终回答, 本次调用的工具名序列）。"""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config or {})
    called: list[str] = []
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            called.append(call["name"])
    return text_of(result["messages"][-1]), called

In [ ]:
# ================================================================
# 第 0 步：最小 agent —— 只有模型 + 循环
# ================================================================
# 基线：create_agent(model, tools=[])。它已经会对话、会调用你给的工具，
# 但**没有任何 harness**：没有文件系统、没有摘要、没有子代理。
# 先让它去读一个文件，看它怎么"无能为力" —— 这就是后面每一步要解决的痛点。
def step_0_minimal(sales_path: str) -> dict:
    print("=" * 70)
    print("第 0 步：最小 agent（模型 + 循环，无 harness）")
    print("=" * 70)

    agent = create_agent(model=model, tools=[])
    answer, called = ask(agent, f"读一下 {sales_path} 这个文件，告诉我一共几行数据。")
    print(f"  可用工具：{tool_names_of([]) or '（无）'}")
    print(f"  回答：{answer[:120]}")
    print(
        "  ↑ 它没法读文件 —— 不是模型不行，是**没给它这个能力**。\n"
        "    （本机模型在这一步还可能把「我要调 read_file」写进正文而不是真的调用工具，\n"
        "      这是模型行为不是代码问题；给它真工具之后就没这个现象了。）\n"
        "    官方的组装教程就是从这里出发，一件件把能力装上去。"
    )
    return {"middleware": []}

In [ ]:
# ================================================================
# 第 1 步：+ 文件系统（FilesystemMiddleware + 后端）
# ================================================================
# 官方的第二个组件：隔离后端 + 文件工具。
# 本文件用 `FilesystemBackend(root_dir=临时目录, virtual_mode=True)`：
#   · virtual_mode=True → agent 看到的是以 "/" 开头的**虚拟路径**，映射到 root_dir；
#   · 换成 StateBackend 则文件存在图状态里（课案 03_deepagents/03 讲过后端选型）；
#   · 官方教程用的是 LangSmithSandbox（真隔离，但要 LANGSMITH_API_KEY），
#     本仓库不用 LangSmith，所以这里的"隔离"仅靠目录隔离 —— 演示够用，生产按需换。
def step_1_filesystem(sales_path: str) -> dict:
    print("\n" + "=" * 70)
    print("第 1 步：+ 文件系统（FilesystemMiddleware + FilesystemBackend）")
    print("=" * 70)

    from deepagents.backends import FilesystemBackend
    from deepagents.middleware.filesystem import FilesystemMiddleware

    backend = FilesystemBackend(root_dir=WORKDIR, virtual_mode=True)
    middleware = [FilesystemMiddleware(backend=backend, tools="all")]
    agent = create_agent(model=model, tools=[], middleware=middleware)

    answer, called = ask(agent, f"读一下 {sales_path} 这个文件，告诉我一共几行数据。")
    print(f"  可用工具：{tool_names_of(middleware)}")
    print(f"  本次调用的工具：{called}")
    print(f"  回答：{answer[:140]}")
    print(
        "  ↑ 这一步只加了一个中间件，agent 立刻多了 ls / read_file / write_file / grep 等工具。\n"
        "    注意工具列表是**中间件带进来的**，不是你一个个注册的 —— 这就是 harness 的含义。"
    )
    return {"middleware": middleware, "backend": backend}

In [ ]:
# ================================================================
# 第 2 步：+ 摘要压缩（SummarizationMiddleware）
# ================================================================
# 痛点是长会话：历史越堆越多，早晚撞上下文窗口。
# SummarizationMiddleware 在**触发条件命中**时，把旧历史交给模型压成一份摘要，
# 用摘要替换原文 —— 之后每轮请求都轻装上阵。
#   参数（实测签名）：trigger=("messages", N) / ("tokens", N) / ("fraction", 0.8)
#                    keep=("messages", M) 控制压缩后保留最近多少条
# 本 Demo 直接**喂一段长历史**（不必真聊 30 轮），一次 invoke 就能看到压缩发生。
def step_2_summarization(built: dict) -> dict:
    print("\n" + "=" * 70)
    print("第 2 步：+ 摘要压缩（SummarizationMiddleware）")
    print("=" * 70)

    middleware = list(built["middleware"]) + [
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 12),   # 历史超过 12 条就压缩
            keep=("messages", 4),       # 压缩后保留最近 4 条
        )
    ]
    agent = create_agent(model=model, tools=[], middleware=middleware)

    # 伪造一段 16 条的长历史（8 轮一问一答），主题固定便于观察摘要是否保留要点
    history = []
    for index in range(1, 9):
        history.append({"role": "user", "content": f"第 {index} 个问题：请记住重点 {index}。"})
        history.append({"role": "assistant", "content": f"好的，我记住了重点 {index}。"})
    history.append({"role": "user", "content": "我一共让你记住了几个重点？"})
    print(f"  投喂历史消息数：{len(history)}（触发阈值 12）")

    result = agent.invoke({"messages": history})
    final_messages = result["messages"]
    print(f"  运行结束后消息数：{len(final_messages)} ← 比投喂的少，说明历史被压缩替换了")
    print(f"  回答：{text_of(final_messages[-1])[:140]}")
    print(
        "  ↑ 关键点：**压缩是自动的**，由中间件在合适时机做，业务代码一行都不用改。\n"
        "    代价是多一次「总结」模型调用；触发阈值与保留条数就是你调节成本/记忆的旋钮\n"
        "    （课案 03_deepagents/14 官方补充篇还讲了另一条路线：把大块内容卸载到文件系统）。"
    )
    built["middleware"] = middleware
    return built

In [ ]:
# ================================================================
# 第 3 步：+ 技能（SkillsMiddleware，渐进披露）
# ================================================================
# 痛点：领域规则（行业口径、报表模板、公司规范）全塞进 system prompt 会撑爆上下文，
# 而且大部分轮次根本用不到。
# SkillsMiddleware 的做法：只把技能的**名字与描述**放进提示词，正文在需要时用
# read_file 去读（渐进披露）—— 与课案 08_skills 章讲的是同一个机制，
# 区别只是课案用 create_deep_agent 的 skills= 参数一键装上，这里手动装。
def step_3_skills(built: dict, skills_dir: Path) -> dict:
    print("\n" + "=" * 70)
    print("第 3 步：+ 技能（SkillsMiddleware + 渐进披露）")
    print("=" * 70)

    from deepagents.middleware.skills import SkillsMiddleware

    # 技能就是磁盘上的目录：skills/<名字>/SKILL.md（与课案 08_skills 同规范）
    skill_file = skills_dir / "sales-report" / "SKILL.md"
    skill_file.parent.mkdir(parents=True, exist_ok=True)
    skill_file.write_text(
        "---\n"
        "name: sales-report\n"
        "description: 销售报表分析规范：必须按「总行数 → 总金额 → 异常值」三段式回答\n"
        "---\n\n"
        "# 销售报表分析技能\n\n"
        "分析销售数据时，严格按以下三段式输出：\n"
        "1. **总行数**：数据一共多少行；\n"
        "2. **总金额**：amount 列求和；\n"
        "3. **异常值**：是否存在负数或空值。\n",
        encoding="utf-8",
    )

    middleware = list(built["middleware"]) + [
        # sources 的形状（官方 docstring 实测）：裸路径 str，或 **(路径, 标签) 元组** ——
        # 注意顺序是「路径在前、标签在后」（写成 (标签, 路径) 会把标签当路径，
        # 报 `Path '标签': path_not_found`，本文件踩过）。
        # 路径指向**技能目录的父目录**；virtual_mode 下要写虚拟路径（"/skills"）。
        SkillsMiddleware(backend=built["backend"], sources=[("/skills", "本课示例")])
    ]
    agent = create_agent(model=model, tools=[], middleware=middleware)

    # 先看技能带来的工具/提示词变化，再实际跑一次
    answer, called = ask(agent, "按销售报表分析技能的要求，分析一下这份数据。")
    print(f"  可用工具：{tool_names_of(middleware)}")
    print(f"  本次调用的工具：{called}")
    print(f"  回答：{answer[:160]}")
    print(
        "  ↑ 技能正文并没有被塞进 system prompt：agent 先看到「有哪些技能」，\n"
        "    判断相关后再用 read_file 把 SKILL.md 读进来（可以看上面的工具调用序列）。\n"
        "    这就是渐进披露：**上下文只花在真正用到的知识上**。"
    )
    built["middleware"] = middleware
    return built

In [ ]:
# ================================================================
# 第 4 步：+ 子代理（SubAgentMiddleware）
# ================================================================
# 痛点：主线程要同时干很多事（查数据、画图、写结论），中间过程互相干扰、上下文互相污染。
# 子代理的做法：把"画图"这类活儿交给一个**独立上下文的 worker**，主 agent 只拿到结论。
# 机制上就是给主 agent 加一个 task 工具，由它发话让子代理干活。
def step_4_subagent(built: dict) -> dict:
    print("\n" + "=" * 70)
    print("第 4 步：+ 子代理（SubAgentMiddleware）")
    print("=" * 70)

    from deepagents.middleware.subagents import SubAgent, SubAgentMiddleware

    middleware = list(built["middleware"]) + [
        SubAgentMiddleware(
            backend=built["backend"],     # 子代理与主代理共享文件系统
            subagents=[
                # SubAgent 的字段（实测）：name / description / tools / model /
                # middleware / interrupt_on / skills / permissions /
                # response_format / system_prompt / mode
                SubAgent(
                    name="chart-designer",
                    description="负责把数据结论整理成图表建议（只需要给结论，不要过程）",
                    system_prompt=(
                        "你是图表设计师。用户会给你一组数据结论，"
                        "你只回一句「推荐图表 + 理由」，不要复述数据。"
                    ),
                    # 手动挂 SubAgentMiddleware 时 **model 是必填**（实测：
                    # 不传直接 ValueError: SubAgent 'x' must specify 'model'）；
                    # 用 create_deep_agent 的 subagents= 时它会自动继承主模型。
                    model=model,
                    tools=[],
                )
            ],
        )
    ]
    agent = create_agent(
        model=model,
        tools=[],
        middleware=middleware,
        # 明确要求委派：不然模型经常"自己顺手做了"（实测第一次跑就是这样，
        # 工具序列里没有 task —— 委派是**模型决策**，提示词要说到位）。
        system_prompt=(
            "把「出图表建议」这件事交给 chart-designer 子代理完成（用 task 工具），"
            "自己不要代替它做这件事。"
        ),
    )

    answer, called = ask(
        agent,
        "先用 sales-report 技能读数据，再把结论交给 chart-designer 出图表建议，最后汇总给我。",
    )
    print(f"  可用工具：{tool_names_of(middleware)}")
    print(f"  本次调用的工具：{called}")
    print(f"  回答：{answer[:200]}")
    if "task" in called:
        print("  ✔ 本次确实发生了委派（工具序列里出现了 task）")
    else:
        print("  （本次模型选择自己做、没调 task —— 委派与否是模型决策；本 Demo 已在\n"
              "   系统提示里明确要求委派，多跑一次通常能命中）")
    print(
        "  ↑ 注意工具列表里多了一个 `task`：主 agent 通过它把活儿派给子代理，\n"
        "    子代理跑完只把**结论**回传 —— 中间过程不进主线程上下文。\n"
        "    课案 03_deepagents/12 讲的是 create_deep_agent 的 subagents= 用法，\n"
        "    本步把它背后的中间件摊开给你看。"
    )
    built["middleware"] = middleware
    return built

In [ ]:
# ================================================================
# 收尾：手装的这套 vs create_deep_agent 的默认栈
# ================================================================
def show_comparison(built: dict) -> None:
    print("\n" + "=" * 70)
    print("对照：我手装的 4 件 vs create_deep_agent 的默认栈")
    print("=" * 70)

    hand_made = [type(item).__name__ for item in built["middleware"]]
    print("  本文件手装的（按组装顺序）：")
    for index, name in enumerate(hand_made, start=1):
        print(f"    {index}. {name}")

    # 默认栈清单来自 deepagents.create_deep_agent 的文档字符串（本机 0.7.13 实测提取）
    default_stack = [
        "SkillsMiddleware（传了 skills 才有）",
        "FilesystemMiddleware",
        "SubAgentMiddleware",
        "SummarizationMiddleware",
        "PatchToolCallsMiddleware（补全被打断的工具调用）",
        "AsyncSubAgentMiddleware（传了 async subagents 才有）",
        "MemoryMiddleware（传了 memory 才有）",
        "HumanInTheLoopMiddleware（传了 interrupt_on 才有）",
        "（另按模型厂商自动加 Prompt Caching 中间件）",
    ]
    print("\n  create_deep_agent 默认装的：")
    for index, name in enumerate(default_stack, start=1):
        print(f"    {index}. {name}")

    print(
        "\n  ↑ 对照结论：本文件装的 4 件（文件系统 / 摘要 / 技能 / 子代理）\n"
        "    与默认栈的前 4 件**完全对应** —— 官方说的「默认 harness」就是这几件。\n"
        "    剩下两件课案没讲过的，也在这里点出来：\n"
        "      · PatchToolCallsMiddleware：历史里若有「发起工具调用但没结果」的消息，\n"
        "        它会补一条占位结果，避免模型因协议不完整而报错（长会话/中断恢复时常见）；\n"
        "      · Prompt Caching 中间件：按模型厂商自动挂（Anthropic/Bedrock/Fireworks），\n"
        "        用同厂模型时能显著省钱，本机网关用不到。\n"
        "    所以「自己组装」的实际收益是：**能只装需要的几件，也能替换任意一件**。"
    )

In [ ]:
with tempfile.TemporaryDirectory(prefix="harness_demo_") as tmp:
    WORKDIR = Path(tmp)
    # 准备一份极小的"销售数据"，供 agent 读取
    sales = WORKDIR / "sales.csv"
    sales.write_text("order_id,amount\nA001,120\nA002,80\nA003,-30\n", encoding="utf-8")
    skills_dir = WORKDIR / "skills"
    print(f"演示工作目录：{WORKDIR}\n")

    sales_virtual_path = "/sales.csv"      # virtual_mode 下 agent 看到的是虚拟路径
    built = step_0_minimal(sales_virtual_path)
    built = step_1_filesystem(sales_virtual_path)
    built = step_2_summarization(built)
    built = step_3_skills(built, skills_dir)
    built = step_4_subagent(built)
    show_comparison(built)

print("\n全部步骤执行完毕（临时目录已清理）。")

### 预期输出

```text
第 0 步：最小 agent（模型 + 循环，无 harness）
  可用工具：（无）
  回答： 我无法直接读取文件...（模型「无能为力」，因为没有文件工具）

第 1 步：+ 文件系统
  可用工具：['delete','edit_file','execute','glob','grep','ls','read_file','write_file']
  本次调用的工具：['read_file']
  回答： 4 行（含表头）/ 3 条数据

第 2 步：+ 摘要压缩
  投喂历史消息数：17（触发阈值 12）
  运行结束后消息数：6 ← 比投喂的少，说明历史被压缩替换了

第 3 步：+ 技能（SkillsMiddleware + 渐进披露）
  本次调用的工具：['ls', 'read_file']  （agent 自己读 SKILL.md）
  回答： 按「总行数 → 总金额 → 异常值」三段式作答

第 4 步：+ 子代理（SubAgentMiddleware）
  可用工具：[..., 'task']
  本次调用的工具：[..., 'task']  （若模型选择委派）

对照：我手装的 4 件 vs create_deep_agent 的默认栈
  1. FilesystemMiddleware
  2. SummarizationMiddleware
  3. SkillsMiddleware
  4. SubAgentMiddleware
```

> ⚠️ 本格输出的 AI 回复、第 0/4 步「读不读文件 / 委不委派」的模型决策、以及临时目录名
> `harness_demo_xxxx`（随机）每次运行都不同；稳定的是第 1~3 步的「可用工具列表 / 消息数变化」。
> 上面的正文是本次实测值。

注意第 0、4 步涉及模型决策（读不读文件、委不委派），输出会因模型而异；
第 1~3 步的结构（可用工具列表、消息数变化）是稳定可复现的。

## 小结

- **人工审核**（HITL）= 危险工具调用前挂起，`interrupt_on` 指定哪些工具、允许哪些决策；
  必须配 checkpointer，恢复用同一个 `thread_id` + `Command(resume=...)`；
- **钩子**分两类：节点式（`before/after_*`，改 state、可 `jump_to`）与
  包裹式（`wrap_*`，洋葱嵌套，可短路/重试）；`dynamic_prompt` 是第 7 个便捷封装；
- **内置中间件** = 钩子的官方封装，别自己再写一遍：摘要 / 限次 / 重试 / 待办 / 清理 /
  人工审核（课案 7 个）+ 工具异常 / 模型降级 / 限工具次数 / PII 脱敏 / 工具模拟（补充 5 个）；
- **上下文工程**的三类控制（Model/Tool/Life-cycle）× 三种数据源（Runtime/State/Store），
  机制全部落在中间件钩子上；
- **自己组装 harness** = 从最小 agent 起步，按需加文件系统 / 摘要 / 技能 / 子代理，
  对应 `create_deep_agent` 默认栈的前四件。

## 常见坑

1. **忘了声明 `can_jump_to`**：钩子里返回 `{"jump_to": "end"}` 却没在装饰器上写
   `can_jump_to=["end"]`，运行时会直接报错（框架没预先连好那条边）。
2. **把包裹式钩子当节点用**：`wrap_model_call` 拿到的是 `request`（即将发出的请求），
   不是完整 state；要改 state 请用节点式钩子 `return dict`。
3. **`compile()` 后 `input()` 无头必崩**：notebook 里 `input()` 会 `EOFError`，
   按模板用 `scripted_input` 临时替换 `builtins.input`，用完必须兜底（审核轮数由模型决定）。
4. **洋葱顺序决定组合语义**：`middleware` 列表第一个 = 最外层；「先重试后兜底」必须
   `ToolError` 在外、`ToolRetry` 在内（顺序反了重试根本不会发生）。
5. **同名参数不同类型**：`SummarizationMiddleware` 的 `trigger=("messages", N)` 是元组，
   `ClearToolUsesEdit` 的 `trigger=N` 是整数 —— 抄参数别互相带。
6. **ModelCallLimit 默认 `exit_behavior='end'`，ToolCallLimit 默认 `'continue'`**，
   兄弟俩默认值相反，别记混。
7. **动态 `response_format` 只能收窄到创建时声明过的 schema**：临时引入新 schema 会抛
   `ValueError: ToolStrategy specifies tool 'X' which wasn't declared...`，
   正确姿势是创建 agent 时用 `ToolStrategy(Union[...])` 一次声明齐。
8. **自定义状态 schema 必须继承 `MessagesState`**：裸 `TypedDict` 自声明 `messages: list`
   会丢掉 `add_messages` reducer，消息被覆盖、agent 循环判不出终止条件（实测无限调模型）。

## 官方链接

- Middleware（内置中间件 / 钩子）：<https://docs.langchain.com/oss/python/langchain/middleware>
- 上下文工程总纲：<https://docs.langchain.com/oss/python/langchain/context-engineering>
- 自己组装 harness（Build a data analysis agent from scratch）：
  <https://docs.langchain.com/oss/python/langchain/deep-agent-from-scratch>
- 课案「结构化输出」（`08_结构化输出_jxsd.py`，第 4 节 Demo 2 的前置知识）：
  <https://docs.langchain.com/oss/python/langchain/structured-output>